# Train the possession sequence model

Trains on the open DFL and Metrica sample matches only. **Do not upload the competition match data to Kaggle** -- it is not needed here and its terms forbid redistribution.

Before running: in the right-hand panel set **Accelerator = GPU** and **Internet = On**. Then choose *Run All*. Takes roughly 3 hours, so use *Save Version -> Save & Run All*.

When it finishes, download `sequence.pt` from the Output panel (`/kaggle/working/sequence.pt`) and copy the printed scores back to the team.

In [ ]:
import os
os.makedirs('/kaggle/working/repo/possession/vendor', exist_ok=True)
os.makedirs('/kaggle/working/repo/scripts', exist_ok=True)
os.chdir('/kaggle/working/repo')
import torch; print('GPU available:', torch.cuda.is_available())

In [ ]:
%%writefile possession/__init__.py
"""Ball-possession estimation from automatic tracking data (More than a Hack 2027, Round 1)."""


In [ ]:
%%writefile possession/baseline.py
"""The organisers' baseline on a Match: nearest player to a seen ball, then a majority vote.

Kept only as the reference line on our benchmark, so every gain is measured against the same
thing the published floor (spellF1 0.2433) measures.
"""

import numpy as np

from .match import H, A, D, X, HOME


def label_match(match, window=50):
    n = match.n_frames
    codes = np.full(n, -1, dtype=np.int8)
    seen = np.flatnonzero(~np.isnan(match.bx))
    for a in range(0, len(seen), 20000):
        rows = seen[a:a + 20000]
        dist = np.hypot(match.px[rows] - match.bx[rows, None], match.py[rows] - match.by[rows, None])
        dist = np.where(np.isnan(dist), np.inf, dist)
        codes[rows] = np.where(match.pteam[dist.argmin(axis=1)] == HOME, H, A)

    home = np.concatenate(([0], np.cumsum(codes == H)))
    away = np.concatenate(([0], np.cumsum(codes == A)))
    index = np.arange(n)
    low, high = np.maximum(0, index - window), np.minimum(n, index + window + 1)
    n_home, n_away = home[high] - home[low], away[high] - away[low]
    labels = np.full(n, D, dtype=np.int8)
    some = (n_home + n_away) > 0
    labels[some] = np.where(n_home[some] >= n_away[some], H, A)

    out = np.full(n, X, dtype=np.int8)
    for start, stop in match.runs:
        out[start:stop] = labels[start:stop]
    return out


In [ ]:
%%writefile possession/decode.py
"""Turn per-frame class probabilities into clean possession segments.

A per-frame classifier flickers, and the ranking metric punishes every flicker as an invented
possession. Viterbi decoding finds the single most likely label sequence when every change of
label has a price, so a change only happens where the evidence pays for it.

The price can vary from frame to frame. With a boundary model, a change is cheap where that
model says a change is happening and expensive everywhere else, which lets short possessions
through without letting flicker through.
"""

import numpy as np

from .match import H, A, D, X


def viterbi(log_prob, cost_team, cost_dead):
    """Most likely H/A/D path.

    log_prob is [n, 3]. cost_team and cost_dead are the prices, in log-probability units, of a
    team change and of a dead/alive change on entering each frame: scalars or length-n arrays.
    """
    n = len(log_prob)
    emit = log_prob.tolist()
    team = np.broadcast_to(np.asarray(cost_team, dtype=np.float64), (n,)).tolist()
    dead = np.broadcast_to(np.asarray(cost_dead, dtype=np.float64), (n,)).tolist()

    h, a, d = emit[0]
    back = np.zeros((n, 3), dtype=np.int8)
    back_rows = []
    for t in range(1, n):
        ct, cd = team[t], dead[t]
        e = emit[t]
        # Best predecessor for each state; ties keep the current state.
        bh, ph = h, 0
        if a - ct > bh:
            bh, ph = a - ct, 1
        if d - cd > bh:
            bh, ph = d - cd, 2
        ba, pa = a, 1
        if h - ct > ba:
            ba, pa = h - ct, 0
        if d - cd > ba:
            ba, pa = d - cd, 2
        bd, pd = d, 2
        if h - cd > bd:
            bd, pd = h - cd, 0
        if a - cd > bd:
            bd, pd = a - cd, 1
        h, a, d = bh + e[0], ba + e[1], bd + e[2]
        back_rows.append((ph, pa, pd))
    if back_rows:
        back[1:] = np.array(back_rows, dtype=np.int8)

    path = np.empty(n, dtype=np.int8)
    path[-1] = int(np.argmax([h, a, d]))
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def change_costs(change_prob, scale, radius=8, floor=1e-4):
    """Per-frame switching prices from the boundary model's probabilities.

    The boundary model says a change lies within `radius` frames; spread over that window, the
    chance it falls on this exact frame is p / (2 * radius). The price is its negative log.
    """
    per_frame = np.clip(change_prob / (2.0 * radius), floor, 1.0)
    return -scale * np.log(per_frame)


def decode_match(prob, runs, switch_team=16.0, switch_dead=16.0, change=None, scale=None):
    """Decode each run on its own. prob is [n_frames, 3] over H, A, D.

    With `change` ([n_frames, 3] over no change / team change / dead change) and `scale`, the
    switching prices follow the boundary model instead of being constant.
    """
    labels = np.full(len(prob), X, dtype=np.int8)
    for start, stop in runs:
        log_prob = np.log(np.clip(prob[start:stop], 1e-6, 1.0))
        if change is None:
            cost_team, cost_dead = switch_team, switch_dead
        else:
            cost_team = change_costs(change[start:stop, 1], scale)
            cost_dead = change_costs(change[start:stop, 2], scale)
        labels[start:stop] = viterbi(log_prob, cost_team, cost_dead)
    return labels


def drop_short(labels, runs, min_frames):
    """Hand possessions shorter than `min_frames` to the label before them."""
    out = labels.copy()
    for start, stop in runs:
        window = out[start:stop]
        breaks = np.flatnonzero(window[1:] != window[:-1]) + 1
        bounds = np.concatenate(([0], breaks, [len(window)]))
        for i in range(1, len(bounds) - 1):
            a, b = int(bounds[i]), int(bounds[i + 1])
            if window[a] in (H, A) and b - a < min_frames:
                window[a:b] = window[a - 1]
    return out


In [ ]:
%%writefile possession/degrade.py
"""Make clean external tracking look like the competition's automatic tracking.

The external matches are corrected tracking: ball on every frame, every player on every frame.
The competition ATD is not: the ball is unseen on ~41% of graded frames and players drop in and
out. A method tuned on clean data would be tuned for a different problem, so every external
match is degraded first.

The defaults were calibrated on 11 Nov 2026 against summary statistics of the competition feed
(scripts/atd_stats.py). That feed is smoothed, not jittery: short losses of the ball or of a
player are filled in by the tracker, so what remains missing is long stretches (median about
10 s for the ball), and the seen ball often moves in straight filled-in lines.

Ball loss is not uniform. It is lost when it is in the air, when it is in a crowd and when the
ball is out of play, so the loss process is a two-state chain whose rates depend on those.
All randomness comes from one seeded generator, so a degraded match is reproducible.
"""

from dataclasses import dataclass, replace

import numpy as np

from .match import D, X


AIR_LOST = 3.0


@dataclass
class Noise:
    ball_missing: float = 0.41      # target share of unseen ball frames inside the runs
    ball_mean_gap: float = 150.0    # mean length of a raw loss, frames, in open play
    ball_fill: int = 75             # losses up to this long are filled with a straight line
    ball_smooth: int = 9            # moving-average window on the seen ball, frames
    air_factor: float = 4.0         # how much likelier the ball is lost above head height
    crowd_factor: float = 2.5       # ... with 3+ players within 3 m
    feet_factor: float = 2.5        # ... with a player within 1 m (ball hidden by the body)
    dead_factor: float = 3.0        # ... while the ball is out of play
    ball_jitter: float = 0.01       # metres, per-frame gaussian
    ball_drift: float = 0.45         # metres, slowly wandering position error (per axis)
    ball_false: float = 0.0015      # share of seen frames replaced by a false detection
    player_missing: float = 0.07    # share of player observations dropped
    player_mean_gap: float = 150.0  # mean length of a raw player dropout, frames
    player_fill: int = 60           # dropouts up to this long are filled (player stays tracked)
    player_jitter: float = 0.003    # metres
    player_smooth: int = 7          # moving-average window on player positions, frames
    cam_height: float = 28.0        # metres above the pitch; 0 switches the effect off
    cam_back: float = 30.0          # metres behind the touchline
    cam_along: float = 0.0          # metres from the halfway line, along the pitch
    cam_side: int = -1              # -1: camera beyond y = 0, +1: beyond y = width
    air_lost: float = AIR_LOST      # metres: a ball higher than this is never detected
    out_seen: float = 0.4           # share of the ball's trips beyond the lines that are seen
    dead_hidden: float = 0.45       # share of stoppages where the ball is never seen at all


def random_noise(rng):
    """A noise setting drawn from a wide range.

    The real tracking's exact noise is not known in advance, so training matches are degraded
    with many different settings. A model that only works for one of them is not one we want.
    """
    return Noise(
        ball_missing=float(rng.uniform(0.30, 0.50)),
        ball_mean_gap=float(rng.uniform(60.0, 350.0)),
        ball_fill=int(rng.integers(25, 126)),
        ball_smooth=int(rng.integers(1, 16)),
        air_factor=float(rng.uniform(2.0, 6.0)),
        crowd_factor=float(rng.uniform(1.5, 4.0)),
        feet_factor=float(rng.uniform(1.0, 4.0)),
        dead_factor=float(rng.uniform(1.5, 5.0)),
        ball_jitter=float(rng.uniform(0.0, 0.05)),
        ball_drift=float(rng.uniform(0.0, 1.2)),
        ball_false=float(rng.uniform(0.0, 0.004)),
        player_missing=float(rng.uniform(0.03, 0.12)),
        player_mean_gap=float(rng.uniform(60.0, 400.0)),
        player_fill=int(rng.integers(20, 101)),
        player_jitter=float(rng.uniform(0.0, 0.03)),
        player_smooth=int(rng.integers(1, 12)),
        cam_height=float(rng.uniform(12.0, 30.0)),
        cam_back=float(rng.uniform(15.0, 50.0)),
        cam_along=float(rng.uniform(-15.0, 15.0)),
        cam_side=int(rng.choice((-1, 1))),
        air_lost=float(rng.uniform(2.5, 8.0)),
        out_seen=float(rng.uniform(0.1, 1.0)),
        dead_hidden=float(rng.uniform(0.15, 0.75)))


def _chain(rng, lose, regain):
    """Two-state chain with per-frame probabilities of losing and regaining. True = unseen."""
    n = len(lose)
    draws = rng.random(n)
    missing = np.zeros(n, dtype=bool)
    state = False
    for i in range(n):
        if state:
            if draws[i] < regain[i]:
                state = False
        elif draws[i] < lose[i]:
            state = True
        missing[i] = state
    return missing


def _ball_missing(match, noise, rng, forced):
    n = match.n_frames
    risk = np.ones(n, dtype=np.float64)
    if match.bz is not None:
        risk[np.nan_to_num(match.bz) > 2.0] *= noise.air_factor
    dist = np.hypot(match.px - match.bx[:, None], match.py - match.by[:, None])
    risk[(np.nan_to_num(dist, nan=np.inf) < 3.0).sum(axis=1) >= 3] *= noise.crowd_factor
    risk[(np.nan_to_num(dist, nan=np.inf) < 1.0).any(axis=1)] *= noise.feet_factor
    if match.labels is not None:
        risk[match.labels == D] *= noise.dead_factor

    in_runs = np.zeros(n, dtype=bool)
    for start, stop in match.runs:
        in_runs[start:stop] = True

    # Riskier frames are both easier to lose and slower to recover. The base loss rate is found
    # by bisection so the unseen share lands on the target whatever the match looks like.
    regain = np.clip((1.0 / noise.ball_mean_gap) / np.sqrt(risk), 0.0, 1.0)
    low, high = 1e-5, 0.5
    seed = int(rng.integers(0, 2 ** 31 - 1))
    for _ in range(18):
        mid = np.sqrt(low * high)
        lose = np.clip(mid * np.sqrt(risk), 0.0, 1.0)
        missing = _long_only(_chain(np.random.default_rng(seed), lose, regain) | forced,
                             noise.ball_fill)
        if missing[in_runs].mean() > noise.ball_missing:
            high = mid
        else:
            low = mid
    return missing


def _segments(mask):
    edges = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    return zip(np.flatnonzero(edges == 1).tolist(), np.flatnonzero(edges == -1).tolist())


def _long_only(mask, shortest):
    """Drop the True stretches of at most `shortest` frames: the tracker fills those in."""
    edges = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    short = (stops - starts) <= shortest
    delta = np.zeros(len(mask) + 1, dtype=np.int32)
    np.add.at(delta, starts[short], 1)
    np.add.at(delta, stops[short], -1)
    return mask & (np.cumsum(delta[:-1]) == 0)


def _fill_line(values, keep_missing):
    """Straight line through every missing frame that is not in `keep_missing`."""
    known = ~np.isnan(values)
    if known.sum() < 2:
        return values
    index = np.arange(len(values))
    filled = np.interp(index, index[known], values[known]).astype(values.dtype)
    filled[keep_missing] = np.nan
    return filled


def _parallax(match, noise):
    """Where a single camera draws the ball on the ground when it is in the air.

    The tracking has no height: an airborne ball is placed where the camera's line of
    sight through it meets the pitch, which is further from the camera than the ball
    really is. A ball 3 m up, seen from a stand 20 m high and 50 m away, lands about 9 m
    off. Comparing the competition's two camera feeds shows exactly this: when they
    disagree about the ball it is across the pitch, by metres.
    """
    bx, by = match.bx.copy(), match.by.copy()
    if match.bz is None or noise.cam_height <= 0:
        return bx, by
    length, width = match.pitch
    cam_x = length / 2.0 + noise.cam_along
    cam_y = -noise.cam_back if noise.cam_side < 0 else width + noise.cam_back
    height = np.clip(np.nan_to_num(match.bz), 0.0, 0.7 * noise.cam_height)
    stretch = height / (noise.cam_height - height)
    bx += (stretch * (bx - cam_x)).astype(np.float32)
    by += (stretch * (by - cam_y)).astype(np.float32)
    return bx, by


def _drift(rng, n, size, fps):
    """Smooth wandering error with standard deviation `size`, changing over a few seconds."""
    if size <= 0:
        return np.zeros(n, dtype=np.float32)
    window = int(3 * fps)
    walk = np.convolve(rng.normal(0.0, 1.0, n + window), np.hanning(window), mode="same")[:n]
    return (walk / walk.std() * size).astype(np.float32)


def _smooth(values, window):
    """Moving average that ignores missing frames and keeps them missing."""
    if window <= 1:
        return values
    known = ~np.isnan(values)
    kernel = np.ones(window)
    total = np.convolve(np.where(known, values, 0.0), kernel, mode="same")
    count = np.convolve(known.astype(np.float64), kernel, mode="same")
    out = (total / np.maximum(count, 1.0)).astype(values.dtype)
    out[~known] = np.nan
    return out


def simulate_dead_ball(match, noise, rng):
    """Replace the ball during stoppages with what a camera would plausibly see.

    The external tracking freezes the ball at one point for the whole of every stoppage. Real
    automatic tracking does not: the ball rolls on after going dead, disappears, then comes
    back being carried to the restart spot and is put down before the restart. A model trained
    on the frozen ball would learn "motionless ball = dead" as a perfect rule that does not
    exist in the competition data, so the frozen stretch is rebuilt here from the real player
    movement: roll-out, unseen, carried by the player who takes the restart, placed.
    """
    bx, by = match.bx.copy(), match.by.copy()
    fps = match.fps
    for start, stop in match.runs:
        dead = match.labels[start:stop] == D
        for a, b in _segments(dead):
            a, b = a + start, b + start
            bx[a:b], by[a:b] = np.nan, np.nan
            if rng.random() < noise.dead_hidden:
                continue
            # Roll-out: keep the last live velocity, slowing down.
            if a - 6 >= start and match.labels[a - 6] != D:
                vx = (match.bx[a - 1] - match.bx[a - 6]) / 5.0
                vy = (match.by[a - 1] - match.by[a - 6]) / 5.0
                k = min(b - a, int(rng.integers(10, 50)))
                decay = np.cumsum(np.exp(-np.arange(1, k + 1) / float(fps)))
                bx[a:a + k] = match.bx[a - 1] + vx * decay
                by[a:a + k] = match.by[a - 1] + vy * decay
            else:
                k = 0
            if b >= stop:
                continue  # the half ends dead: no restart to carry the ball to
            # Carried to the restart spot by whoever is nearest to it when play resumes.
            spot_x, spot_y = match.bx[b], match.by[b]
            dist = np.hypot(match.px[b] - spot_x, match.py[b] - spot_y)
            if np.all(np.isnan(dist)):
                continue
            taker = int(np.nanargmin(dist))
            still = int(rng.uniform(0.5, 4.0) * fps)
            carry = int(rng.uniform(3.0, 12.0) * fps)
            place = max(a + k, b - still)
            pick = max(a + k, place - carry)
            bx[pick:place] = match.px[pick:place, taker] + 0.3
            by[pick:place] = match.py[pick:place, taker]
            bx[place:b], by[place:b] = spot_x, spot_y
    return bx, by


def degrade(match, noise=None, seed=0):
    noise = noise or Noise()
    rng = np.random.default_rng(seed)
    n, slots = match.px.shape
    length, width = match.pitch

    seen_x, seen_y = _parallax(match, noise)
    if match.labels is not None:
        dead_x, dead_y = simulate_dead_ball(match, noise, rng)
        alive = ~np.isnan(dead_x) & (match.labels != D) & (match.labels != X)
        dead_x[alive], dead_y[alive] = seen_x[alive], seen_y[alive]
        if match.bz is not None:
            high = alive & (np.nan_to_num(match.bz) > noise.air_lost)
            dead_x[high], dead_y[high] = np.nan, np.nan
    else:
        dead_x, dead_y = seen_x, seen_y
    # The tracker looks at the pitch: most of what the ball does beyond the lines is not seen.
    beyond = (dead_x < 0) | (dead_x > length) | (dead_y < 0) | (dead_y > width)
    for a, b in _segments(beyond):
        if rng.random() > noise.out_seen:
            dead_x[a:b], dead_y[a:b] = np.nan, np.nan
    missing = _ball_missing(match, noise, rng, np.isnan(dead_x))
    bx = dead_x + rng.normal(0.0, noise.ball_jitter, n).astype(np.float32)
    by = dead_y + rng.normal(0.0, noise.ball_jitter, n).astype(np.float32)
    bx += _drift(rng, n, noise.ball_drift, match.fps)
    by += _drift(rng, n, noise.ball_drift, match.fps)
    false = rng.random(n) < noise.ball_false
    false &= ~np.isnan(bx)
    bx[false] = rng.uniform(0.0, length, int(false.sum()))
    by[false] = rng.uniform(0.0, width, int(false.sum()))
    # `missing` holds only the long losses. Everything shorter is drawn as a straight line
    # between the sightings on either side, which is what the competition tracker does.
    bx[np.isnan(dead_x)] = np.nan
    by[np.isnan(dead_x)] = np.nan
    lost = _chain(rng, np.full(n, 0.01), np.full(n, 0.10)) & ~missing
    bx[lost], by[lost] = np.nan, np.nan
    bx = _smooth(_fill_line(bx, missing), noise.ball_smooth)
    by = _smooth(_fill_line(by, missing), noise.ball_smooth)

    px = match.px + rng.normal(0.0, noise.player_jitter, (n, slots)).astype(np.float32)
    py = match.py + rng.normal(0.0, noise.player_jitter, (n, slots)).astype(np.float32)
    regain = np.full(n, 1.0 / noise.player_mean_gap)
    lose = np.full(n, noise.player_missing / (1.0 - noise.player_missing) / noise.player_mean_gap)
    for slot in range(slots):
        gone = _long_only(_chain(rng, lose, regain), noise.player_fill)
        px[:, slot] = _smooth(px[:, slot], noise.player_smooth)
        py[:, slot] = _smooth(py[:, slot], noise.player_smooth)
        px[gone, slot] = np.nan
        py[gone, slot] = np.nan

    return replace(match, name=match.name + "-noisy", px=px, py=py, bx=bx, by=by, bz=None)


In [ ]:
%%writefile possession/diagnostics.py
"""Summary statistics of a tracking feed. Aggregates only -- never rows, frames or positions.

Used to check that the degraded external matches look like the competition tracking: the same
table is printed for both and compared by eye.
"""

import numpy as np

from .features import remove_ball_teleports


def _stretches(mask):
    edges = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    return np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)


def _pct(values, points=(10, 25, 50, 75, 90, 99)):
    if not len(values):
        return "n/a"
    return " ".join("{:.4g}".format(v) for v in np.percentile(values, points))


def tracking_stats(match):
    """Dict of printable lines describing the feed inside the match's play runs."""
    fps = match.fps
    length, width = match.pitch
    rows = np.concatenate([np.arange(a, b) for a, b in match.runs])
    out = {}

    seen = ~np.isnan(match.bx)
    clean_x, _ = remove_ball_teleports(match.bx, match.by, fps)
    out["frames in play runs"] = "{}".format(len(rows))
    out["ball unseen share"] = "{:.3f}".format(1.0 - seen[rows].mean())
    out["ball false detections removed (share of seen)"] = "{:.4f}".format(
        float((seen & np.isnan(clean_x))[rows].sum()) / max(1, int(seen[rows].sum())))

    gaps, seen_runs = [], []
    for a, b in match.runs:
        gaps.append(_stretches(~seen[a:b]))
        seen_runs.append(_stretches(seen[a:b]))
    gaps, seen_runs = np.concatenate(gaps), np.concatenate(seen_runs)
    out["ball unseen stretches: count"] = "{}".format(len(gaps))
    out["ball unseen stretch length, frames (p10 p25 p50 p75 p90 p99)"] = _pct(gaps)
    out["ball seen stretch length, frames (p10 p25 p50 p75 p90 p99)"] = _pct(seen_runs)
    for lo, hi in ((1, 5), (5, 25), (25, 75), (75, 250), (250, 10 ** 9)):
        inside = gaps[(gaps >= lo) & (gaps < hi)]
        out["unseen frames in stretches of {}-{} frames (share of unseen)".format(
            lo, hi if hi < 10 ** 9 else "inf")] = "{:.3f}".format(
                float(inside.sum()) / max(1, int(gaps.sum())))

    bx, by = match.bx[rows], match.by[rows]
    off = (bx < 0) | (bx > length) | (by < 0) | (by > width)
    out["seen ball outside the pitch lines (share of seen)"] = "{:.4f}".format(
        float(np.nansum(off)) / max(1, int(seen[rows].sum())))

    known = seen[rows]
    out["seen ball beyond the touchline y < 0 (share of seen)"] = "{:.4f}".format(
        float(np.nansum(by < 0)) / max(1, int(known.sum())))
    out["seen ball beyond the touchline y > width (share of seen)"] = "{:.4f}".format(
        float(np.nansum(by > width)) / max(1, int(known.sum())))
    out["seen ball beyond a goal line (share of seen)"] = "{:.4f}".format(
        float(np.nansum((bx < 0) | (bx > length))) / max(1, int(known.sum())))

    # Jitter: spread of the second difference, which a smooth path keeps near zero.
    second = match.bx[2:] - 2 * match.bx[1:-1] + match.bx[:-2]
    out["ball jitter proxy, metres (median |2nd difference|)"] = "{:.4f}".format(
        float(np.nanmedian(np.abs(second))))
    step = np.hypot(np.diff(match.bx), np.diff(match.by)) * fps
    out["ball speed, m/s (p10 p25 p50 p75 p90 p99)"] = _pct(step[~np.isnan(step)])

    # A tracker that fills short losses draws straight lines at constant speed. How much of the
    # seen ball is exactly that tells us how much of it is filled in rather than detected.
    bend = np.hypot(second, match.by[2:] - 2 * match.by[1:-1] + match.by[:-2])
    pace = step[1:]
    ok = ~np.isnan(bend) & ~np.isnan(pace)
    out["ball bend per frame, metres (p10 p25 p50 p75 p90 p99)"] = _pct(bend[ok])
    out["ball exactly still (share of seen)"] = "{:.3f}".format(
        float((pace[ok] < 0.01).mean()) if ok.any() else 0.0)
    for limit in (0.001, 0.01):
        straight = _stretches(ok & (bend < limit) & (pace >= 0.5))
        out["ball moving dead straight, bend < {} m (share of seen)".format(limit)] =             "{:.3f}".format(float(straight.sum()) / max(1, int(ok.sum())))
        out["  ... stretch length, frames, bend < {} m (p10 p25 p50 p75 p90 p99)".format(limit)] =             _pct(straight)

    # How the ball behaves when it is seen: a ball that is often motionless or often right at a
    # player's feet says a lot about what the feed shows during stoppages.
    step_rows = np.hypot(match.bx[rows] - match.bx[rows - 1], match.by[rows] - match.by[rows - 1]) * fps
    moving = ~np.isnan(step_rows)
    out["seen ball slower than 0.5 m/s (share of consecutive sightings)"] = "{:.3f}".format(
        float((step_rows[moving] < 0.5).mean()) if moving.any() else 0.0)
    out["seen ball slower than 2 m/s (share of consecutive sightings)"] = "{:.3f}".format(
        float((step_rows[moving] < 2.0).mean()) if moving.any() else 0.0)

    present = ~np.isnan(match.px[rows])
    count = present.sum(axis=1)
    out["players tracked per frame (p10 p25 p50 p75 p90 p99)"] = _pct(count)
    out["players tracked per frame: mean"] = "{:.2f}".format(count.mean())
    out["frames with more than 22 tracked"] = "{:.4f}".format(float((count > 22).mean()))
    for team in (0, 1):
        cols = match.pteam == team
        out["team {} tracked per frame: mean".format(team)] = "{:.2f}".format(
            present[:, cols].sum(axis=1).mean())
    out["track slots"] = "{} (team0 {}, team1 {})".format(
        len(match.pteam), int((match.pteam == 0).sum()), int((match.pteam == 1).sum()))

    lengths, holes = [], []
    for a, b in match.runs:
        there = ~np.isnan(match.px[a:b])
        for slot in range(there.shape[1]):
            lengths.append(_stretches(there[:, slot]))
            inner = _stretches(~there[:, slot])
            holes.append(inner)
    lengths, holes = np.concatenate(lengths), np.concatenate(holes)
    out["player track pieces: count"] = "{}".format(len(lengths))
    out["player track piece length, frames (p10 p25 p50 p75 p90 p99)"] = _pct(lengths)
    out["player hole length, frames (p10 p25 p50 p75 p90 p99)"] = _pct(holes)

    second = match.px[2:] - 2 * match.px[1:-1] + match.px[:-2]
    out["player jitter proxy, metres (median |2nd difference|)"] = "{:.4f}".format(
        float(np.nanmedian(np.abs(second[::10]))))
    bend = np.hypot(second[::10], (match.py[2:] - 2 * match.py[1:-1] + match.py[:-2])[::10])
    out["player bend per frame, metres (p10 p25 p50 p75 p90 p99)"] = _pct(bend[~np.isnan(bend)])

    near = np.hypot(match.px[rows] - match.bx[rows, None], match.py[rows] - match.by[rows, None])
    nearest = np.nanmin(np.where(np.isnan(near), np.inf, near), axis=1)
    nearest = nearest[np.isfinite(nearest)]
    out["distance ball to nearest player, metres (p10 p25 p50 p75 p90 p99)"] = _pct(nearest)
    for limit in (0.5, 1.0, 2.0):
        out["ball within {} m of a player (share of seen)".format(limit)] = "{:.3f}".format(
            float((nearest < limit).mean()) if len(nearest) else 0.0)

    # Player movement, which is how stoppages show up without any ball at all.
    from .features import player_speeds
    speeds = player_speeds(match.px, match.py, fps)[rows]
    median = np.nanmedian(speeds, axis=1)
    median = median[~np.isnan(median)]
    out["median player speed per frame, m/s (p10 p25 p50 p75 p90 p99)"] = _pct(median)
    out["frames where players are walking, median < 1.2 m/s (share)"] = "{:.3f}".format(
        float((median < 1.2).mean()) if len(median) else 0.0)
    seen_rows = seen[rows]
    walking = np.nan_to_num(np.nanmedian(speeds, axis=1), nan=9.0) < 1.2
    out["ball unseen share while players are walking"] = "{:.3f}".format(
        float(1.0 - seen_rows[walking].mean()) if walking.any() else 0.0)
    out["ball unseen share while players are not walking"] = "{:.3f}".format(
        float(1.0 - seen_rows[~walking].mean()) if (~walking).any() else 0.0)
    return out


def print_stats(title, stats):
    print("\n" + title)
    for key, value in stats.items():
        print("  {:<70} {}".format(key, value))


In [ ]:
%%writefile possession/engine.py
"""The rule-based possession engine: football rules written as code, no training.

Three decisions, made in this order:

  1. DEAD OR ALIVE. The ball is dead only on positive evidence -- it is seen off the pitch, it
     sits still, the players have all slowed to a walk, or it has been unseen for a long time.
     An unseen ball on its own is never evidence: that is the baseline's main mistake.
  2. WHO IS ON THE BALL. Within reach of exactly one team's nearest player.
  3. WHO HAS POSSESSION. A team takes possession only by showing control: staying on the ball
     for `control` frames, or being on it twice in a row (a completed pass). A lone touch --
     deflection, block, lost header -- changes nothing. The change is dated to the first touch.
     Between touches the ball stays with the last team, as the competition rules say.
"""

from dataclasses import dataclass

import numpy as np

from .match import H, A, D, X


@dataclass
class Rules:
    reach: float = 2.0          # metres: a player this close to the ball is "on" it
    margin: float = 0.4         # metres the nearest opponent must be further away than that
    trust: int = 30             # frames from a sighting within which a filled position is trusted
    bridge: int = 8             # join two spells on the ball by the same team this close together
    min_touch: int = 3          # ignore spells on the ball shorter than this
    control: int = 20           # frames on the ball that prove control on their own
    pass_window: int = 100      # second touch by the same team within this many frames = control
    out_margin: float = 0.5     # metres beyond a line before a seen ball counts as out
    still_speed: float = 1.0    # m/s below which the ball counts as stationary
    still_frames: int = 50      # ... for this long
    walk_speed: float = 1.3     # m/s median player speed that looks like a stoppage
    walk_frames: int = 75       # ... for this long
    lost_frames: int = 150      # ball unseen this long means it left play
    dead_merge: int = 50        # join dead stretches closer than this
    dead_min: int = 60          # drop dead stretches shorter than this
    restart_speed: float = 4.0  # m/s: a seen ball this fast means play is on again


def _runs_of(mask):
    """[(start, stop)] of True stretches."""
    edges = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    return list(zip(np.flatnonzero(edges == 1).tolist(), np.flatnonzero(edges == -1).tolist()))


def _sustained(mask, length):
    """True where mask has held for at least `length` consecutive frames (whole stretch marked)."""
    out = np.zeros(len(mask), dtype=bool)
    for a, b in _runs_of(mask):
        if b - a >= length:
            out[a:b] = True
    return out


def _close_open(mask, merge, minimum):
    """Join True stretches separated by less than `merge`, then drop those under `minimum`."""
    mask = ~_drop_short(~mask, merge)
    return _drop_short(mask, minimum)


def _drop_short(mask, minimum):
    out = mask.copy()
    for a, b in _runs_of(mask):
        if b - a < minimum and a > 0 and b < len(mask):
            out[a:b] = False
    return out


def dead_mask(f, rules, start, stop):
    s = slice(start, stop)
    seen = f.seen[s]
    out = seen & (f.out[s] > rules.out_margin)
    still = _sustained((f.speed[s] < rules.still_speed) & (f.edge[s] <= 2), rules.still_frames)
    walk = _sustained(f.walk[s] < rules.walk_speed, rules.walk_frames)
    lost = f.gap[s] >= rules.lost_frames
    dead = out | still | walk | lost
    # Play is certainly on wherever a seen ball is moving fast inside the pitch.
    dead &= ~(seen & (f.speed[s] > rules.restart_speed) & (f.out[s] == 0) & ~walk)
    return _close_open(dead, rules.dead_merge, rules.dead_min)


def on_ball(f, rules, start, stop):
    """Per frame: 0 home on the ball, 1 away, -1 nobody (or position not trustworthy)."""
    s = slice(start, stop)
    dh, da = f.d_home[s], f.d_away[s]
    who = np.full(stop - start, -1, dtype=np.int8)
    trusted = f.edge[s] <= rules.trust
    who[trusted & (dh < rules.reach) & (da > dh + rules.margin)] = H
    who[trusted & (da < rules.reach) & (dh > da + rules.margin)] = A
    return who


def touches(who, rules):
    """[(start, stop, team)] spells on the ball, near-adjacent same-team spells joined."""
    spells = []
    for team in (H, A):
        mask = who == team
        mask = ~_drop_short(~mask, rules.bridge)
        # Bridging must not swallow the other team's time on the ball.
        mask &= who != (A if team == H else H)
        spells += [(a, b, team) for a, b in _runs_of(mask) if b - a >= rules.min_touch]
    spells.sort()
    return spells


def possession(f, rules, start, stop, dead):
    """Label one run. Returns int8 codes for frames start..stop."""
    n = stop - start
    labels = np.full(n, X, dtype=np.int8)
    labels[dead] = D
    who = on_ball(f, rules, start, stop)
    nearer = np.where(f.d_home[start:stop] <= f.d_away[start:stop], H, A)

    previous_owner = None
    for a, b in _runs_of(~dead):
        spells = [(s + a, e + a, team) for s, e, team in touches(who[a:b], rules)]
        if not spells:
            # Nobody clearly on the ball in this whole passage: majority of "nearer team".
            owner = int(np.bincount(nearer[a:b], minlength=2).argmax())
            labels[a:b] = owner
            previous_owner = owner
            continue

        owner = spells[0][2]
        cut = a
        for i, (s, e, team) in enumerate(spells):
            if team == owner:
                continue
            controlled = (e - s) >= rules.control
            if not controlled and i + 1 < len(spells):
                nxt = spells[i + 1]
                controlled = nxt[2] == team and nxt[0] - e <= rules.pass_window
            if controlled:
                labels[cut:s] = owner
                owner, cut = team, s
        labels[cut:b] = owner
        previous_owner = owner
    return labels


def label_match(match, f, rules=None):
    rules = rules or Rules()
    labels = np.full(match.n_frames, X, dtype=np.int8)
    for start, stop in match.runs:
        dead = dead_mask(f, rules, start, stop)
        labels[start:stop] = possession(f, rules, start, stop, dead)
    return labels


In [ ]:
%%writefile possession/features.py
"""Per-frame measurements shared by the rule engine and the learned model.

Nothing here assumes a player column is a person: every player quantity is reduced over the
columns of a team (nearest, count within a radius, median speed), so identity switches and
fragmented tracks in the automatic tracking cannot break it.
"""

from dataclasses import dataclass, field

import numpy as np

from . import shape as team_shape
from .match import HOME, AWAY

MAX_BALL_SPEED = 45.0     # m/s; anything faster between two sightings is a false detection
MAX_PLAYER_SPEED = 12.0   # m/s; faster means the slot jumped to another person
SPEED_LAG = 5             # frames used for finite-difference speeds
CHUNK = 20000


@dataclass
class Features:
    seen: np.ndarray        # ball observed on this frame (after removing false detections)
    bx: np.ndarray          # ball position with gaps filled by straight lines inside each run
    by: np.ndarray
    gap: np.ndarray         # length in frames of the unseen stretch this frame is in, 0 if seen
    edge: np.ndarray        # frames to the nearest sighting, 0 if seen
    speed: np.ndarray       # ball speed, m/s
    accel: np.ndarray       # magnitude of ball velocity change, m/s per frame-pair
    d_home: np.ndarray      # distance from ball to nearest home / away player, metres
    d_away: np.ndarray
    n_home: np.ndarray      # players of each team within 3 m of the ball
    n_away: np.ndarray
    out: np.ndarray         # metres the ball is beyond the nearest pitch line, 0 inside
    walk: np.ndarray        # median speed of all tracked players, m/s
    n_players: np.ndarray   # tracked players on the frame
    fast: np.ndarray        # share of tracked players moving faster than 3 m/s
    speed_p90: np.ndarray   # 90th percentile of player speed, m/s
    speed_mean: np.ndarray  # mean player speed, m/s
    spread_x: np.ndarray    # standard deviation of player positions along / across the pitch
    spread_y: np.ndarray
    drift: np.ndarray       # speed of the centroid of all players, m/s
    shape: dict = field(default_factory=dict)   # per-team shape, see shape.py


def remove_ball_teleports(bx, by, fps, max_speed=MAX_BALL_SPEED):
    """Drop sightings that cannot be reached from both neighbouring sightings."""
    bx, by = bx.copy(), by.copy()
    for _ in range(3):
        idx = np.flatnonzero(~np.isnan(bx))
        if len(idx) < 3:
            break
        step = np.hypot(np.diff(bx[idx]), np.diff(by[idx])) / (np.diff(idx) / float(fps))
        too_fast = step > max_speed
        bad = np.zeros(len(idx), dtype=bool)
        bad[1:-1] = too_fast[:-1] & too_fast[1:]
        if not bad.any():
            break
        bx[idx[bad]] = np.nan
        by[idx[bad]] = np.nan
    return bx, by


def fill_ball(bx, by, runs):
    """Straight-line fill of unseen stretches, inside each run only. Returns x, y, gap, edge."""
    fx, fy = bx.copy(), by.copy()
    gap = np.zeros(len(bx), dtype=np.int32)
    edge = np.zeros(len(bx), dtype=np.int32)
    for start, stop in runs:
        window = np.arange(start, stop)
        known = window[~np.isnan(bx[start:stop])]
        if not len(known):
            gap[start:stop] = stop - start
            edge[start:stop] = stop - start
            continue
        fx[start:stop] = np.interp(window, known, bx[known])
        fy[start:stop] = np.interp(window, known, by[known])
        # Position of each frame relative to the sightings either side of it.
        after = np.searchsorted(known, window, side="left")
        before = after - 1
        prev_seen = np.where(before >= 0, known[np.clip(before, 0, len(known) - 1)], start - 1)
        next_seen = np.where(after < len(known), known[np.clip(after, 0, len(known) - 1)], stop)
        missing = np.isnan(bx[start:stop])
        gap[start:stop] = np.where(missing, next_seen - prev_seen - 1, 0)
        edge[start:stop] = np.where(missing, np.minimum(window - prev_seen, next_seen - window), 0)
    return fx, fy, gap, edge


def player_speeds(px, py, fps, lag=SPEED_LAG):
    """Per-slot speed by finite difference, NaN where the slot is unseen or jumped."""
    speed = np.full(px.shape, np.nan, dtype=np.float32)
    step = np.hypot(px[lag:] - px[:-lag], py[lag:] - py[:-lag]) * (fps / float(lag))
    step[step > MAX_PLAYER_SPEED] = np.nan
    speed[lag:] = step
    return speed


def compute(match):
    fps = match.fps
    length, width = match.pitch
    n = match.n_frames

    raw_x, raw_y = remove_ball_teleports(match.bx, match.by, fps)
    seen = ~np.isnan(raw_x)
    bx, by, gap, edge = fill_ball(raw_x, raw_y, match.runs)

    # Ball velocity on the filled track, central difference over +/-2 frames.
    vx = np.zeros(n, dtype=np.float32)
    vy = np.zeros(n, dtype=np.float32)
    vx[2:-2] = (bx[4:] - bx[:-4]) * (fps / 4.0)
    vy[2:-2] = (by[4:] - by[:-4]) * (fps / 4.0)
    vx, vy = np.nan_to_num(vx), np.nan_to_num(vy)
    speed = np.hypot(vx, vy)
    accel = np.zeros(n, dtype=np.float32)
    accel[3:-3] = np.hypot(vx[6:] - vx[:-6], vy[6:] - vy[:-6])

    d_home = np.full(n, np.inf, dtype=np.float32)
    d_away = np.full(n, np.inf, dtype=np.float32)
    n_home = np.zeros(n, dtype=np.int8)
    n_away = np.zeros(n, dtype=np.int8)
    home_cols = np.flatnonzero(match.pteam == HOME)
    away_cols = np.flatnonzero(match.pteam == AWAY)
    for a in range(0, n, CHUNK):
        b = min(n, a + CHUNK)
        dist = np.hypot(match.px[a:b] - bx[a:b, None], match.py[a:b] - by[a:b, None])
        dist = np.where(np.isnan(dist), np.inf, dist)
        d_home[a:b] = dist[:, home_cols].min(axis=1)
        d_away[a:b] = dist[:, away_cols].min(axis=1)
        n_home[a:b] = (dist[:, home_cols] < 3.0).sum(axis=1)
        n_away[a:b] = (dist[:, away_cols] < 3.0).sum(axis=1)

    out = np.maximum.reduce([-bx, bx - length, -by, by - width, np.zeros(n, dtype=np.float32)])
    out = np.nan_to_num(out)

    speeds = player_speeds(match.px, match.py, fps)
    n_players = (~np.isnan(match.px)).sum(axis=1)
    with np.errstate(all="ignore"):
        walk = np.nanmedian(speeds, axis=1)
        speed_p90 = np.nanpercentile(speeds, 90, axis=1)
        speed_mean = np.nanmean(speeds, axis=1)
        fast = np.nanmean(np.where(np.isnan(speeds), np.nan, speeds > 3.0), axis=1)
        spread_x = np.nanstd(match.px, axis=1)
        spread_y = np.nanstd(match.py, axis=1)
        cx, cy = np.nanmean(match.px, axis=1), np.nanmean(match.py, axis=1)
    drift = np.zeros(n, dtype=np.float32)
    drift[fps:] = np.hypot(cx[fps:] - cx[:-fps], cy[fps:] - cy[:-fps])
    clean = lambda v: np.nan_to_num(np.asarray(v, dtype=np.float32))  # noqa: E731

    return Features(seen=seen, bx=bx, by=by, gap=gap, edge=edge, speed=speed, accel=accel,
                    d_home=d_home, d_away=d_away, n_home=n_home, n_away=n_away, out=out,
                    walk=clean(walk), n_players=n_players, fast=clean(fast),
                    speed_p90=clean(speed_p90), speed_mean=clean(speed_mean),
                    spread_x=clean(spread_x), spread_y=clean(spread_y), drift=clean(drift),
                    shape=team_shape.compute(match, bx, by))


In [ ]:
%%writefile possession/io_atd.py
"""Load the competition tracking (Metrica ATD, FIFA/EPTS text export) into a Match.

Raw line:  "<frame_id>:<x,y,speed>;<x,y,speed>;...:<ball x,y,...>"  with NaN for a missing
observation. Coordinates are normalised 0-1. frame_id counts from 1; video frames from 0.
"""

import xml.etree.ElementTree as ET

import numpy as np

from .match import Match, HOME, AWAY

# Per-match constants published by the organisers (official games.py / DATA.md).
GAMES = {
    "BAR-ATH": {
        "n_frames": 199319,
        "fps": 25,
        "pitch": (105.3, 68.0),
        # Graded video-frame spans, half-open: the two halves.
        "runs": [(11185, 90652), (110808, 190198)],
    },
}


def slot_teams(meta_path):
    """HOME/AWAY for every track slot. The first team in the metadata is the home side."""
    root = ET.parse(meta_path).getroot()
    team_ids = [team.get("id") for team in root.iter("Team")]
    if len(team_ids) != 2:
        raise ValueError("{}: expected 2 teams, found {}".format(meta_path, len(team_ids)))
    slots = [player.get("teamId") for player in root.iter("Player")]
    if not slots:
        raise ValueError("{}: no Player entries".format(meta_path))
    return np.array([HOME if t == team_ids[0] else AWAY for t in slots], dtype=np.int8)


def _xy(group):
    """(x, y) of an 'x,y,...' group, NaN if the observation is missing or malformed."""
    if not group or group[0] in "Nn":
        return np.nan, np.nan
    parts = group.split(",")
    try:
        return float(parts[0]), float(parts[1])
    except (ValueError, IndexError):
        return np.nan, np.nan


def frame_rate(meta_path, default=25.0):
    """Frames per second stated in the metadata, or the default if it does not say."""
    for node in ET.parse(meta_path).getroot().iter("FrameRate"):
        try:
            return float(node.text)
        except (TypeError, ValueError):
            pass
    return default


def load_feed(meta_path, raw_path, pitch=(105.3, 68.0)):
    """Any ATD feed on its own timeline (e.g. the broadcast recording), as a Match without runs.

    Row i is the feed's own video frame i; nothing here relates it to the tactical video.
    """
    pteam = slot_teams(meta_path)
    n_slots = len(pteam)
    with open(raw_path) as handle:
        n_frames = max((int(line.split(":", 1)[0]) for line in handle if ":" in line), default=0)
    px = np.full((n_frames, n_slots), np.nan, dtype=np.float32)
    py = np.full((n_frames, n_slots), np.nan, dtype=np.float32)
    bx = np.full(n_frames, np.nan, dtype=np.float32)
    by = np.full(n_frames, np.nan, dtype=np.float32)
    with open(raw_path) as handle:
        for line in handle:
            parts = line.rstrip().split(":")
            if len(parts) != 3:
                continue
            row = int(parts[0]) - 1
            if row < 0:
                continue
            for slot, group in enumerate(parts[1].split(";")[:n_slots]):
                px[row, slot], py[row, slot] = _xy(group)
            bx[row], by[row] = _xy(parts[2])
    length, width = pitch
    return Match(name="feed", fps=int(round(frame_rate(meta_path))), pitch=pitch,
                 px=px * length, py=py * width, pteam=pteam,
                 bx=bx * length, by=by * width, runs=[(0, n_frames)])


def load(meta_path, raw_path, game="BAR-ATH"):
    info = GAMES[game]
    pteam = slot_teams(meta_path)
    n_frames, n_slots = info["n_frames"], len(pteam)
    length, width = info["pitch"]

    px = np.full((n_frames, n_slots), np.nan, dtype=np.float32)
    py = np.full((n_frames, n_slots), np.nan, dtype=np.float32)
    bx = np.full(n_frames, np.nan, dtype=np.float32)
    by = np.full(n_frames, np.nan, dtype=np.float32)

    with open(raw_path) as handle:
        for line in handle:
            parts = line.rstrip("\n").split(":")
            if len(parts) != 3:
                continue
            row = int(parts[0]) - 1
            if not 0 <= row < n_frames:
                continue
            for slot, group in enumerate(parts[1].split(";")[:n_slots]):
                px[row, slot], py[row, slot] = _xy(group)
            bx[row], by[row] = _xy(parts[2])

    return Match(name=game, fps=info["fps"], pitch=info["pitch"],
                 px=px * length, py=py * width, pteam=pteam,
                 bx=bx * length, by=by * width, runs=list(info["runs"]))


In [ ]:
%%writefile possession/io_dfl.py
"""Load the open DFL / Sportec matches (Bassek et al., Scientific Data 2025, CC-BY 4.0).

These are OTHER games, used to develop and train on. The raw tracking carries a per-frame
BallStatus (in play or not) and BallPossession (home or away), which is the same kind of signal
the competition ground truth is derived from -- so each match gives a full H/A/D label track.

The position file is ~400 MB of one-Frame-per-line XML, so it is streamed with regexes rather
than parsed as a tree.
"""

import glob
import os
import re
import xml.etree.ElementTree as ET

import numpy as np

from .match import Match, H, A, D, X, HOME, AWAY

FPS = 25
SECTIONS = ("firstHalf", "secondHalf")
GAP_FRAMES = 1000  # ungraded filler between the halves, so no window ever spans both

_FRAMESET = re.compile(r'<FrameSet ([^>]*)>')
_ATTR = re.compile(r'(\w+)="([^"]*)"')
_N = re.compile(r' N="(\d+)"')
_X = re.compile(r' X="([^"]*)"')
_Y = re.compile(r' Y="([^"]*)"')
_Z = re.compile(r' Z="([^"]*)"')
_POSS = re.compile(r' BallPossession="(\d)"')
_STATUS = re.compile(r' BallStatus="(\d)"')


def match_ids(folder):
    paths = sorted(glob.glob(os.path.join(folder, "DFL_04_03_positions_raw_observed_*.xml")))
    return [os.path.basename(p)[:-4].rsplit("-", 1)[1] for p in paths]


def _one(folder, prefix, match_id):
    hits = glob.glob(os.path.join(folder, "{}_*{}.xml".format(prefix, match_id)))
    if len(hits) != 1:
        raise IOError("expected one {} file for {}, found {}".format(prefix, match_id, len(hits)))
    return hits[0]


def _teams(folder, match_id):
    root = ET.parse(_one(folder, "DFL_02_01_matchinformation", match_id)).getroot()
    general = next(root.iter("General"))
    env = next(root.iter("Environment"))
    pitch = (float(env.get("PitchX")), float(env.get("PitchY")))
    return general.get("HomeTeamId"), general.get("GuestTeamId"), pitch


def load(folder, match_id):
    home_id, away_id, pitch = _teams(folder, match_id)

    # First pass collects per (section, object) arrays keyed by frame number.
    tracks = []  # (section, team or "BALL", frame numbers, x, y, [z, poss, status])
    current = None
    with open(_one(folder, "DFL_04_03_positions_raw_observed", match_id), encoding="utf-8") as fh:
        for line in fh:
            if line.startswith("<Frame "):
                if current is None:
                    continue
                current["n"].append(int(_N.search(line).group(1)))
                current["x"].append(float(_X.search(line).group(1)))
                current["y"].append(float(_Y.search(line).group(1)))
                if current["team"] == "BALL":
                    current["z"].append(float(_Z.search(line).group(1)))
                    current["poss"].append(int(_POSS.search(line).group(1)))
                    current["status"].append(int(_STATUS.search(line).group(1)))
                continue
            found = _FRAMESET.search(line)
            if found:
                attrs = dict(_ATTR.findall(found.group(1)))
                team = attrs.get("TeamId")
                if attrs.get("GameSection") in SECTIONS and team in (home_id, away_id, "BALL"):
                    current = {"section": attrs["GameSection"], "team": team,
                               "person": attrs.get("PersonId"),
                               "n": [], "x": [], "y": [], "z": [], "poss": [], "status": []}
                    tracks.append(current)
                else:
                    current = None  # referees and anything outside the two halves

    balls = {t["section"]: t for t in tracks if t["team"] == "BALL"}
    spans, offset, runs = {}, 0, []
    for section in SECTIONS:
        numbers = np.array(balls[section]["n"])
        first, last = int(numbers.min()), int(numbers.max())
        spans[section] = (first, offset)
        runs.append((offset, offset + last - first + 1))
        offset += last - first + 1 + GAP_FRAMES
    n_frames = offset - GAP_FRAMES

    # One column per person: a player keeps a column across both halves.
    people = []
    for t in tracks:
        if t["team"] != "BALL" and (t["team"], t["person"]) not in people:
            people.append((t["team"], t["person"]))
    column = {key: i for i, key in enumerate(people)}

    px = np.full((n_frames, len(people)), np.nan, dtype=np.float32)
    py = np.full((n_frames, len(people)), np.nan, dtype=np.float32)
    bx = np.full(n_frames, np.nan, dtype=np.float32)
    by = np.full(n_frames, np.nan, dtype=np.float32)
    bz = np.full(n_frames, np.nan, dtype=np.float32)
    labels = np.full(n_frames, X, dtype=np.int8)

    half_length, half_width = pitch[0] / 2.0, pitch[1] / 2.0
    for t in tracks:
        first, base = spans[t["section"]]
        rows = np.array(t["n"]) - first + base
        keep = (rows >= 0) & (rows < n_frames)
        rows = rows[keep]
        xs = np.array(t["x"], dtype=np.float32)[keep] + half_length
        ys = np.array(t["y"], dtype=np.float32)[keep] + half_width
        if t["team"] == "BALL":
            bx[rows], by[rows] = xs, ys
            bz[rows] = np.array(t["z"], dtype=np.float32)[keep]
            status = np.array(t["status"])[keep]
            poss = np.array(t["poss"])[keep]
            labels[rows] = np.where(status == 0, D, np.where(poss == 1, H, A))
        else:
            col = column[(t["team"], t["person"])]
            px[rows, col], py[rows, col] = xs, ys

    pteam = np.array([HOME if team == home_id else AWAY for team, _ in people], dtype=np.int8)
    return Match(name="DFL-" + match_id, fps=FPS, pitch=pitch, px=px, py=py, pteam=pteam,
                 bx=bx, by=by, bz=bz, runs=runs, labels=labels)


In [ ]:
%%writefile possession/io_metrica.py
"""Load Metrica Sports' public sample games into a Match, with labels built from their events.

Source: https://github.com/metrica-sports/sample-data (three anonymised games, 25 fps tracking
of players and ball, plus synchronised event data). Games 1 and 2 are in Metrica's CSV format,
game 3 in the FIFA EPTS format (the same format as the competition tracking) with JSON events.

There is no per-frame possession in this data, so the labels are derived from the events:

- **Dead ball** starts at a stoppage event -- the ball going out (at the moment it leaves the
  pitch), a shot that goes out or in, a foul, an offside or a hand ball -- and ends at the
  restart that follows (a set piece, or a goal kick). If play carries on instead (advantage),
  there is no stoppage.
- **Team** is whoever made the last on-the-ball event: a pass, carry, set piece, recovery, shot,
  or the touch that lost the ball. A loose ball stays with the last team, as in the
  competition's definition. One-touch spells are then handed back by `relabel.controlled`,
  exactly as for the other external matches.

Stretches of 2 s or more with no ball in the tracking are also dead: in this data the ball is
removed exactly when play stops, so those are stoppages the events do not mention.

Event times are marked by people and are good to a few frames, so these labels are a little
less precise in time than ones derived from tracking.
"""

import csv
import json
import os
import xml.etree.ElementTree as ET

import numpy as np

from .match import Match, H, A, D, X, HOME, AWAY

PITCH = (105.0, 68.0)
FPS = 25
CONTROL = {"PASS", "CARRY", "SET PIECE", "RECOVERY", "SHOT", "BALL LOST"}
IGNORED = {"CARD", "CHALLENGE", "FAULT RECEIVED", "BALL OUT"}
ADVANTAGE_GAP = 5 * FPS   # a live event within 5 s of a foul means play went on
TAIL = 2 * FPS            # frames of play kept after the last event of a half


def _is_stop(event):
    kind, sub = event["type"], event["sub"]
    if kind == "BALL OUT":
        return True
    if kind == "SHOT" and ("OUT" in sub or "GOAL" in sub):
        return True
    if kind == "FAULT RECEIVED":
        return True
    if kind == "CHALLENGE" and "FAULT" in sub and "ADVANTAGE" not in sub:
        return True
    if kind == "BALL LOST" and ("OFFSIDE" in sub or "HAND BALL" in sub):
        return True
    return False


def _is_restart(event):
    return event["type"] == "SET PIECE" or "GOAL KICK" in event["sub"]


def _stop_frame(event):
    """The ball leaves play at the end of a ball-out or a shot, at the start of a foul."""
    if event["type"] in ("BALL OUT", "SHOT") and event["end"] > event["start"]:
        return event["end"]
    return event["start"]


def labels_from_events(events, n_frames, periods):
    """[n_frames] int8 labels and the runs (one per half) they cover."""
    labels = np.full(n_frames, X, dtype=np.int8)
    runs = []
    for period, (first, last) in sorted(periods.items()):
        mine = sorted((e for e in events if e["period"] == period), key=lambda e: e["start"])
        kick = next((e for e in mine if e["type"] == "SET PIECE" and "KICK OFF" in e["sub"]), None)
        if kick is None or not mine:
            continue
        start = max(first, kick["start"])
        stop = min(last + 1, max(e["end"] for e in mine) + TAIL, n_frames)
        if stop <= start:
            continue
        runs.append((start, stop))

        # Team: forward fill of the last on-the-ball event.
        team = np.full(stop - start, -1, dtype=np.int8)
        for e in mine:
            if e["type"] in CONTROL and start <= e["start"] < stop:
                team[e["start"] - start] = e["team"]
        current = kick["team"]
        for i in range(len(team)):
            if team[i] >= 0:
                current = team[i]
            team[i] = current
        part = np.where(team == HOME, H, A).astype(np.int8)

        # Dead ball: from a stoppage to the restart that follows it.
        plays = [e for e in mine if e["type"] not in IGNORED]
        for e in mine:
            if not _is_stop(e):
                continue
            a = _stop_frame(e)
            following = next((p for p in plays if p is not e and p["start"] >= a), None)
            if following is None:
                b = stop
            elif _is_restart(following) or following["start"] - a > ADVANTAGE_GAP:
                b = following["start"]
            else:
                continue  # play went on
            a, b = max(a, start), min(b, stop)
            if b > a:
                part[a - start:b - start] = D
        labels[start:stop] = part
    return labels, runs


# --- games 1 and 2: CSV ----------------------------------------------------------------------

def _csv_events(path):
    out = []
    with open(path, newline="") as handle:
        for row in csv.DictReader(handle):
            out.append({"team": HOME if row["Team"].strip().lower() == "home" else AWAY,
                        "type": row["Type"].strip().upper(),
                        "sub": row["Subtype"].strip().upper(),
                        "period": int(row["Period"]),
                        "start": int(row["Start Frame"]) - 1,
                        "end": int(row["End Frame"]) - 1})
    return out


def _csv_tracking(path):
    """(period [n], x [n, players], y [n, players], ball x [n], ball y [n]); frames from 1."""
    with open(path, newline="") as handle:
        reader = csv.reader(handle)
        next(reader), next(reader), next(reader)
        rows = [r for r in reader if r]
    frame = np.array([int(r[1]) for r in rows]) - 1
    values = np.array([[float(v) if v not in ("", "NaN") else np.nan for v in r[3:]] for r in rows],
                      dtype=np.float32)
    n = int(frame.max()) + 1
    period = np.zeros(n, dtype=np.int8)
    period[frame] = [int(r[0]) for r in rows]
    full = np.full((n, values.shape[1]), np.nan, dtype=np.float32)
    full[frame] = values
    players = full[:, :-2]
    return period, players[:, 0::2], players[:, 1::2], full[:, -2], full[:, -1]


def load_csv_game(folder, number):
    stem = os.path.join(folder, "Sample_Game_{}_".format(number))
    period, hx, hy, bx, by = _csv_tracking(stem + "RawTrackingData_Home_Team.csv")
    _, ax, ay, _, _ = _csv_tracking(stem + "RawTrackingData_Away_Team.csv")
    n = min(len(period), len(ax))
    px = np.concatenate([hx[:n], ax[:n]], axis=1)
    py = np.concatenate([hy[:n], ay[:n]], axis=1)
    pteam = np.array([HOME] * hx.shape[1] + [AWAY] * ax.shape[1], dtype=np.int8)
    periods = {}
    for p in (1, 2):
        idx = np.flatnonzero(period[:n] == p)
        if len(idx):
            periods[p] = (int(idx[0]), int(idx[-1]))
    labels, runs = labels_from_events(_csv_events(stem + "RawEventsData.csv"), n, periods)
    return _match("MET-{}".format(number), px, py, pteam, bx[:n], by[:n], runs, labels)


# --- game 3: EPTS + JSON ---------------------------------------------------------------------

def _epts_meta(path):
    root = ET.parse(path).getroot()
    team_of = {p.get("id"): p.get("teamId") for p in root.iter("Player")}
    home_id = next(root.iter("Team")).get("id")
    channels = [(c.get("id"), c.get("playerId")) for c in root.iter("PlayerChannel")
                if c.get("channelId") == "x"]
    pteam = np.array([HOME if team_of[player] == home_id else AWAY for _, player in channels],
                     dtype=np.int8)
    # Which player each group of a line belongs to changes with substitutions: every block of
    # frames lists its own order of player channels.
    slot_of = {channel: i for i, (channel, _) in enumerate(channels)}
    layouts = []
    for spec in root.iter("DataFormatSpecification"):
        order = [slot_of[r.get("playerChannelId")] for r in spec.iter("PlayerChannelRef")
                 if r.get("playerChannelId").endswith("_x")]
        layouts.append((int(spec.get("startFrame")), int(spec.get("endFrame")), order))
    params = {p.findtext("Name"): p.findtext("Value") for p in root.iter("ProviderParameter")}
    periods = {}
    for p, key in ((1, "first_half"), (2, "second_half")):
        a, b = params.get(key + "_start"), params.get(key + "_end")
        if a and b:
            periods[p] = (int(a) - 1, int(b) - 1)
    return pteam, home_id, periods, layouts


def _xy(group):
    if not group or group[0] in "Nn":
        return np.nan, np.nan
    parts = group.split(",")
    try:
        return float(parts[0]), float(parts[1])
    except (ValueError, IndexError):
        return np.nan, np.nan


def load_epts_game(folder, number=3):
    stem = os.path.join(folder, "Sample_Game_{}_".format(number))
    pteam, home_id, periods, layouts = _epts_meta(stem + "metadata.xml")
    slots = len(pteam)
    with open(stem + "tracking.txt") as handle:
        lines = [line.rstrip().split(":") for line in handle if ":" in line]
    n = max(int(parts[0]) for parts in lines)
    px = np.full((n, slots), np.nan, dtype=np.float32)
    py = np.full((n, slots), np.nan, dtype=np.float32)
    bx = np.full(n, np.nan, dtype=np.float32)
    by = np.full(n, np.nan, dtype=np.float32)
    for parts in lines:
        if len(parts) != 3:
            continue
        frame = int(parts[0])
        row = frame - 1
        order = next((o for a, b, o in layouts if a <= frame <= b), None)
        if order is None:
            continue
        for slot, group in zip(order, parts[1].split(";")):
            px[row, slot], py[row, slot] = _xy(group)
        bx[row], by[row] = _xy(parts[2])

    with open(stem + "events.json") as handle:
        raw = json.load(handle)["data"]
    events = []
    for e in raw:
        subs = e.get("subtypes") or []
        subs = [subs] if isinstance(subs, dict) else subs
        events.append({"team": HOME if (e.get("team") or {}).get("id") == home_id else AWAY,
                       "type": e["type"]["name"].strip().upper(),
                       "sub": "-".join(s["name"] for s in subs).upper(),
                       "period": int(e["period"]),
                       "start": int(e["start"]["frame"]) - 1,
                       "end": int(e["end"]["frame"]) - 1})
    labels, runs = labels_from_events(events, n, periods)
    return _match("MET-{}".format(number), px, py, pteam, bx, by, runs, labels)


def _dead_while_unseen(labels, bx, runs, shortest=2 * FPS):
    """Mark as dead every stretch of 2 s or more in which the tracking has no ball.

    In these sample games the ball is removed from the tracking exactly when play stops (it
    disappears within a frame of every stoppage found in the events), so a long stretch without
    a ball is a stoppage the events do not mention, such as play halted for an injury.
    """
    labels = labels.copy()
    for start, stop in runs:
        missing = np.isnan(bx[start:stop])
        edges = np.diff(np.concatenate(([0], missing.astype(np.int8), [0])))
        for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
            if b - a >= shortest:
                labels[start + a:start + b] = D
    return labels


def _match(name, px, py, pteam, bx, by, runs, labels):
    labels = _dead_while_unseen(labels, bx, runs)
    length, width = PITCH
    return Match(name=name, fps=FPS, pitch=PITCH, px=px * length, py=py * width, pteam=pteam,
                 bx=bx * length, by=by * width, runs=runs, labels=labels)


def load(folder, number):
    return load_epts_game(folder, number) if number == 3 else load_csv_game(folder, number)


In [ ]:
%%writefile possession/io_pff.py
"""Load PFF FC's 2022 World Cup data into a Match at 25 fps, with labels built from its events.

Source: PFF FC's free 2022 World Cup dataset (registration required): broadcast tracking at
29.97 fps with every player's position (estimated when off screen) and the ball with its height,
plus detailed event data. Coordinates are centred metres; they are shifted to a corner origin.

Labels, from the event data (all times are seconds of video, so frame = time x fps):

- **Team**: every on-the-ball event (OTB) names the team on the ball; the team is carried
  forward until the next one. A loose ball stays with the last team.
- **Dead ball**: every restart is an on-the-ball event with a set-piece type (throw-in, free
  kick, corner, goal kick, kick-off, penalty). The ball is dead from the stoppage before it --
  the ball-out, goal or foul event if there is one, otherwise the end of the last on-the-ball
  event -- until the restart.
- One-touch spells are handed back by `relabel.controlled`, as for the other external matches.
"""

import bz2
import json
import os

import numpy as np

from .match import Match, H, A, D, X, HOME, AWAY

FPS_OUT = 25
LENGTH, WIDTH = 105.0, 68.0
RESTARTS = {"T", "F", "C", "G", "K", "P"}
STOPS = {"OUT", "G", "FOUL"}
MAX_SLOTS = 20


def _xy(item):
    if not item or item.get("x") is None or item.get("y") is None:
        return np.nan, np.nan
    return float(item["x"]) + LENGTH / 2.0, float(item["y"]) + WIDTH / 2.0


def _tracking(path, fps_in):
    """Arrays at the source frame rate, indexed by source frame number."""
    frames, home, away, ball, height = [], [], [], [], []
    slots = {HOME: {}, AWAY: {}}
    with bz2.open(path, "rt") as handle:
        for line in handle:
            try:
                d = json.loads(line)
            except ValueError:
                continue
            if d.get("frameNum") is None:
                continue
            frames.append(int(d["frameNum"]))
            for side, key, out in ((HOME, "homePlayers", home), (AWAY, "awayPlayers", away)):
                row = {}
                for p in d.get(key) or []:
                    number = p.get("jerseyNum")
                    if number is None:
                        continue
                    index = slots[side].setdefault(str(number), len(slots[side]))
                    if index < MAX_SLOTS:
                        row[index] = _xy(p)
                out.append(row)
            b = d.get("ballsSmoothed")
            if isinstance(b, list):
                b = b[0] if b else None
            ball.append(_xy(b))
            height.append(float(b["z"]) if b and b.get("z") is not None else np.nan)
    frames = np.asarray(frames)
    n = int(frames.max()) + 1
    px = np.full((n, 2 * MAX_SLOTS), np.nan, dtype=np.float32)
    py = np.full((n, 2 * MAX_SLOTS), np.nan, dtype=np.float32)
    bx = np.full(n, np.nan, dtype=np.float32)
    by = np.full(n, np.nan, dtype=np.float32)
    bz = np.full(n, np.nan, dtype=np.float32)
    for i, f in enumerate(frames):
        for index, (x, y) in home[i].items():
            px[f, index], py[f, index] = x, y
        for index, (x, y) in away[i].items():
            px[f, MAX_SLOTS + index], py[f, MAX_SLOTS + index] = x, y
        bx[f], by[f] = ball[i]
        bz[f] = height[i]
    used = [i for i in range(MAX_SLOTS) if i < len(slots[HOME])] + \
           [MAX_SLOTS + i for i in range(MAX_SLOTS) if i < len(slots[AWAY])]
    pteam = np.array([HOME] * min(len(slots[HOME]), MAX_SLOTS) +
                     [AWAY] * min(len(slots[AWAY]), MAX_SLOTS), dtype=np.int8)
    return px[:, used], py[:, used], pteam, bx, by, bz


def _events(path):
    out = []
    for e in json.load(open(path, encoding="utf-8")):
        g = e.get("gameEvents") or {}
        start = e.get("startTime")
        if start is None:
            continue
        end = e.get("endTime") or start
        out.append({"type": g.get("gameEventType"), "setpiece": g.get("setpieceType"),
                    "home": g.get("homeTeam"), "period": g.get("period"),
                    "start": float(start), "end": max(float(end), float(start))})
    return sorted(out, key=lambda e: e["start"])


def labels_from_events(events, meta, n):
    """[n] labels at 25 fps and the runs (one per regular half)."""
    labels = np.full(n, X, dtype=np.int8)
    runs = []
    for period in (1, 2):
        first = meta.get("startPeriod{}".format(period))
        last = meta.get("endPeriod{}".format(period))
        if first is None or last is None:
            # Knockout matches have no period times in the metadata: take them from the
            # kick-off that opens the half and the end-of-period event that closes it.
            kick = "FIRSTKICKOFF" if period == 1 else "SECONDKICKOFF"
            first = next((e["start"] for e in events if e["type"] == kick), None)
            ends = [e["start"] for e in events if e["type"] == "END" and e["period"] == period]
            inside = [e["start"] for e in events if e["period"] == period]
            last = min(ends) if ends else (max(inside) if inside else None)
        if first is None or last is None:
            continue
        start, stop = int(round(first * FPS_OUT)), min(n, int(round(last * FPS_OUT)))
        if stop <= start:
            continue
        runs.append((start, stop))
        mine = [e for e in events if first - 1.0 <= e["start"] <= last + 1.0]

        # Team: forward fill of the team named by each on-the-ball event.
        team = np.full(stop - start, -1, dtype=np.int8)
        for e in mine:
            if e["type"] == "OTB" and e["home"] is not None:
                f = int(round(e["start"] * FPS_OUT)) - start
                if 0 <= f < len(team):
                    team[f] = H if e["home"] else A
        current = next((t for t in team if t >= 0), H)
        for i in range(len(team)):
            if team[i] >= 0:
                current = team[i]
            team[i] = current
        part = team.copy()

        # Dead ball: from the stoppage before every restart to the restart.
        previous_end, stops_since = first, []
        for e in mine:
            if e["type"] in STOPS:
                stops_since.append(e["start"])
                continue
            if e["type"] != "OTB":
                continue
            if e["setpiece"] in RESTARTS and e["start"] > first + 0.5:
                dead_from = min(stops_since) if stops_since else previous_end
                a = max(0, int(round(dead_from * FPS_OUT)) - start)
                b = min(len(part), int(round(e["start"] * FPS_OUT)) - start)
                if b > a:
                    part[a:b] = D
            previous_end, stops_since = e["end"], []
        labels[start:stop] = part
    return labels, runs


def load(folder, match_id):
    meta = json.load(open(os.path.join(folder, "metadata", "{}.json".format(match_id))))[0]
    fps_in = float(meta.get("fps") or 29.97)
    px, py, pteam, bx, by, bz = _tracking(
        os.path.join(folder, "tracking", "{}.jsonl.bz2".format(match_id)), fps_in)
    # Resample to 25 fps: the nearest source frame to every output time.
    n = int(len(bx) / fps_in * FPS_OUT)
    source = np.clip(np.round(np.arange(n) / FPS_OUT * fps_in).astype(np.int64), 0, len(bx) - 1)
    px, py, bx, by, bz = px[source], py[source], bx[source], by[source], bz[source]
    events = _events(os.path.join(folder, "events", "{}.json".format(match_id)))
    labels, runs = labels_from_events(events, meta, n)
    return Match(name="PFF-{}".format(match_id), fps=FPS_OUT, pitch=(LENGTH, WIDTH), px=px, py=py,
                 pteam=pteam, bx=bx, by=by, runs=runs, labels=labels, bz=bz)


In [ ]:
%%writefile possession/match.py
"""One in-memory format for every match, whatever provider it came from.

Everything downstream (features, engine, model) reads a Match and nothing else, so the same code
runs on the competition ATD and on the external matches we benchmark on.

Coordinates are metres with the origin at a pitch corner: x in [0, length], y in [0, width].
Player columns are *slots*, not people: nothing downstream may assume a column follows one
player, because the competition tracking has no identities.
"""

from dataclasses import dataclass, field

import numpy as np

H, A, D, X = 0, 1, 2, 3
LABELS = np.array(["H", "A", "D", "X"])
HOME, AWAY = 0, 1


@dataclass
class Match:
    name: str
    fps: int
    pitch: tuple                  # (length, width) in metres
    px: np.ndarray                # [n_frames, n_slots] float32, NaN where unseen
    py: np.ndarray                # [n_frames, n_slots] float32
    pteam: np.ndarray             # [n_slots] int8, HOME or AWAY
    bx: np.ndarray                # [n_frames] float32, NaN where the ball is unseen
    by: np.ndarray                # [n_frames] float32
    runs: list                    # [(start, stop)] half-open frame spans that are match play
    labels: np.ndarray = None     # [n_frames] int8 H/A/D/X, only for matches with ground truth
    bz: np.ndarray = None         # [n_frames] ball height, external data only (never an input)
    extra: dict = field(default_factory=dict)

    @property
    def n_frames(self):
        return len(self.bx)

    def save(self, path):
        np.savez_compressed(
            path, name=self.name, fps=self.fps, pitch=np.array(self.pitch),
            px=self.px, py=self.py, pteam=self.pteam, bx=self.bx, by=self.by,
            runs=np.array(self.runs, dtype=np.int64),
            labels=self.labels if self.labels is not None else np.array([]),
            bz=self.bz if self.bz is not None else np.array([]))

    @staticmethod
    def load(path):
        z = np.load(path, allow_pickle=False)
        return Match(
            name=str(z["name"]), fps=int(z["fps"]), pitch=tuple(float(v) for v in z["pitch"]),
            px=z["px"], py=z["py"], pteam=z["pteam"], bx=z["bx"], by=z["by"],
            runs=[(int(a), int(b)) for a, b in z["runs"]],
            labels=z["labels"] if z["labels"].size else None,
            bz=z["bz"] if z["bz"].size else None)


In [ ]:
%%writefile possession/model.py
"""The learned model: per-frame H/A/D probabilities from tracking geometry.

Trained only on OTHER matches (open DFL data, degraded to look like the competition tracking).
Each frame is described by what is happening at that instant and by short summaries of the
seconds before and after it -- possession is a property of an interval, so a frame on its own
does not hold the answer. The rule engine's own verdict is one of the inputs: the model learns
where the rules can be trusted and where they cannot.

The two teams are interchangeable, so every prediction is made twice -- once as given and once
with the teams swapped -- and averaged. The model therefore cannot favour "home".
"""

from dataclasses import replace

import numpy as np

from . import engine, shape as team_shape
from .match import H, A, D

OFFSETS = (5, 12, 25, 50)
WINDOWS = (12, 25, 50, 100, 200)
TRUST = 30


def _shift(x, k):
    """x at t+k, with the ends held."""
    out = np.empty_like(x)
    if k > 0:
        out[:-k], out[-k:] = x[k:], x[-1]
    else:
        out[-k:], out[:-k] = x[:k], x[0]
    return out


def _past_future_mean(x, w):
    """Mean of x over the w frames before t and over the w frames after t."""
    c = np.concatenate(([0.0], np.cumsum(x, dtype=np.float64)))
    n = len(x)
    i = np.arange(n)
    lo = np.maximum(0, i - w)
    past = (c[i] - c[lo]) / np.maximum(1, i - lo)
    hi = np.minimum(n, i + 1 + w)
    future = (c[hi] - c[i + 1]) / np.maximum(1, hi - (i + 1))
    return past.astype(np.float32), future.astype(np.float32)


def _since_until(flag, cap=250):
    """Frames since the flag was last true and until it is next true, capped, in seconds."""
    n = len(flag)
    index = np.arange(n)
    last = np.maximum.accumulate(np.where(flag, index, -cap))
    following = np.minimum.accumulate(np.where(flag, index, n + cap)[::-1])[::-1]
    since = np.minimum(index - last, cap) / 25.0
    until = np.minimum(following - index, cap) / 25.0
    return since.astype(np.float32), until.astype(np.float32)


def swap_teams(f):
    """The same features seen from the other side."""
    return replace(f, d_home=f.d_away, d_away=f.d_home, n_home=f.n_away, n_away=f.n_home,
                   shape=team_shape.swap(f.shape))


def base_features(match, f, rules=None, shape=False):
    """{name: [n_frames] float32}: what is happening on each frame, with no time context."""
    length, width = match.pitch
    rule_labels = engine.label_match(match, f, rules)

    dh = np.minimum(f.d_home, 40.0)
    da = np.minimum(f.d_away, 40.0)
    trusted = f.edge <= TRUST
    bx, by = np.nan_to_num(f.bx, nan=length / 2), np.nan_to_num(f.by, nan=width / 2)

    base = {
        "seen": f.seen.astype(np.float32),
        "gap_s": np.minimum(f.gap, 500) / 25.0,
        "edge_s": np.minimum(f.edge, 250) / 25.0,
        "speed": np.minimum(f.speed, 40.0),
        "accel": np.minimum(f.accel, 40.0),
        "d_home": dh,
        "d_away": da,
        "d_min": np.minimum(dh, da),
        "d_diff": np.clip(da - dh, -20.0, 20.0),
        "on_home": (trusted & (dh < 2.0) & (da > dh + 0.4)).astype(np.float32),
        "on_away": (trusted & (da < 2.0) & (dh > da + 0.4)).astype(np.float32),
        "n_home": f.n_home.astype(np.float32),
        "n_away": f.n_away.astype(np.float32),
        "out": np.minimum(f.out, 10.0),
        "to_touchline": np.minimum(by, width - by),
        "to_goal_line": np.minimum(bx, length - bx),
        "walk": f.walk,
        "n_players": f.n_players.astype(np.float32),
        "fast": f.fast,
        "speed_p90": f.speed_p90,
        "speed_mean": f.speed_mean,
        "spread_x": f.spread_x,
        "spread_y": f.spread_y,
        "drift": f.drift,
        "rule_home": (rule_labels == H).astype(np.float32),
        "rule_away": (rule_labels == A).astype(np.float32),
        "rule_dead": (rule_labels == D).astype(np.float32),
    }
    base = {k: np.nan_to_num(np.asarray(v, dtype=np.float32)) for k, v in base.items()}

    # Who was last on the ball, and who is on it next: the plainest evidence of possession
    # for all the frames in between, where nobody is near the ball.
    for side in ("home", "away"):
        since, until = _since_until(base["on_" + side] > 0)
        base["since_on_" + side] = since
        base["until_on_" + side] = until
    base["since_diff"] = base["since_on_away"] - base["since_on_home"]
    base["until_diff"] = base["until_on_away"] - base["until_on_home"]
    since, until = _since_until(base["seen"] > 0)
    base["since_seen"], base["until_seen"] = since, until
    if shape:  # the sequence model's extra inputs; the tree model was trained without them
        for key, values in f.shape.items():
            base["shape_" + key] = values
    return base


def matrix(match, f, rules=None):
    """(features [n_frames, n_features], names): each frame plus summaries of its surroundings."""
    base = base_features(match, f, rules)
    columns, names = [], []
    for name, values in base.items():
        columns.append(values)
        names.append(name)
    for name in ("d_diff", "d_min", "speed", "seen", "on_home", "on_away", "walk"):
        for k in OFFSETS:
            for sign, tag in ((-1, "m"), (1, "p")):
                columns.append(_shift(base[name], sign * k))
                names.append("{}_{}{}".format(name, tag, k))
    for name in ("on_home", "on_away", "seen", "speed", "walk", "out", "d_min",
                 "rule_home", "rule_away", "rule_dead", "fast", "speed_p90", "speed_mean",
                 "spread_x", "spread_y", "drift"):
        for w in WINDOWS:
            past, future = _past_future_mean(base[name], w)
            columns += [past, future]
            names += ["{}_past{}".format(name, w), "{}_next{}".format(name, w)]
    return np.column_stack(columns), names


NO_CHANGE, TEAM_CHANGE, DEAD_CHANGE = 0, 1, 2
BOUNDARY_RADIUS = 8


def boundary_target(truth, runs, radius=BOUNDARY_RADIUS):
    """Per frame: is a change of label close by, and of which kind?

    TEAM_CHANGE where possession passes between the teams within `radius` frames, DEAD_CHANGE
    where the ball goes dead or comes back into play. Trained as a second model so the decoder
    knows *where* a change is plausible, not only which label each frame looks like.
    """
    target = np.zeros(len(truth), dtype=np.int64)
    for start, stop in runs:
        window = truth[start:stop]
        for i in np.flatnonzero(window[1:] != window[:-1]) + 1:
            kind = DEAD_CHANGE if D in (window[i - 1], window[i]) else TEAM_CHANGE
            a, b = max(0, i - radius), min(len(window), i + radius)
            span = target[start + a:start + b]
            span[span == NO_CHANGE] = kind
    return target


def predict(booster, match, f, rules=None, boundary_booster=None):
    """[n_frames, 3] probabilities over H, A, D, averaged over both team orderings.

    With a boundary model, also returns its [n_frames, 3] probabilities over
    no change / team change / dead change.
    """
    x, _ = matrix(match, f, rules)
    x_swapped, _ = matrix(match, swap_teams(f), rules)
    prob = (booster.predict(x) + booster.predict(x_swapped)[:, [A, H, D]]) / 2.0
    if boundary_booster is None:
        return prob
    return prob, (boundary_booster.predict(x) + boundary_booster.predict(x_swapped)) / 2.0


In [ ]:
%%writefile possession/relabel.py
"""Turn touch-level possession labels into controlled-possession labels.

The open DFL matches flip possession on every touch: a blocked pass or a lost header becomes a
half-second possession for the other side. The competition's definition (CHALLENGE.md) does not:
possession changes only when the other team gains *controlled* possession, and a loose or
contested ball stays with the last team that had it.

So before an external match is used for development, an opposing spell that lasts less than
`min_control` frames is handed back to the team that had the ball. A spell that opens a passage
of play (straight after a dead ball) is always kept: a restart has a clear possessor.
"""

import numpy as np

from .match import H, A, D, X


def controlled(labels, runs, min_control):
    out = labels.copy()
    for start, stop in runs:
        window = labels[start:stop]
        breaks = np.flatnonzero(window[1:] != window[:-1]) + 1
        bounds = np.concatenate(([0], breaks, [len(window)]))
        owner = None
        for i in range(len(bounds) - 1):
            a, b = int(bounds[i]), int(bounds[i + 1])
            code = int(window[a])
            if code not in (H, A):
                owner = None  # dead ball: the next spell is a restart
                continue
            if owner is None or code == owner or (b - a) >= min_control:
                owner = code
            else:
                out[start + a:start + b] = owner
    return out


In [ ]:
%%writefile possession/scoring.py
"""Score a label track with the official metric code (vendored unchanged, MIT licence)."""

import numpy as np

from .match import X
from .vendor import possession_metrics as pm


def graded_truth(match):
    """Ground-truth codes with everything outside the match's play runs forced to X."""
    truth = np.full(match.n_frames, X, dtype=np.int8)
    for start, stop in match.runs:
        truth[start:stop] = match.labels[start:stop]
    return truth


def score(truth, pred):
    """The numbers we steer by. spell is the ranking metric."""
    result = pm.evaluate(truth, np.asarray(pred, dtype=np.int8))
    return {
        "spell": result["spell_f1_mean"],
        "macro": result["macro_f1"],
        "n_gt": result["n_gt_spells"],
        "n_pred": result["n_pred_spells"],
        "dur": result["mean_spell_duration_s"],
        "dur_gt": result["gt_mean_spell_duration_s"],
        "alive_dead": result["alive_dead_f1"],
        "team_acc": result["team_acc_both_alive"],
        "f1_D": result["per_class"]["D"]["f1"],
        "spell_by_iou": {k: v["f1"] for k, v in result["spell_f1"].items()},
    }


def line(name, s):
    return ("{:<28} spell {:.4f}  macro {:.4f}  poss {:>4}/{:<4} dur {:5.1f}/{:<5.1f} "
            "alive/dead {:.3f}  team {:.3f}  D-F1 {:.3f}").format(
                name, s["spell"], s["macro"], s["n_pred"], s["n_gt"], s["dur"], s["dur_gt"],
                s["alive_dead"], s["team_acc"], s["f1_D"])


In [ ]:
%%writefile possession/seqmodel.py
"""The sequence model: a small temporal convolutional network over the whole match.

The boosted-tree model sees each frame with hand-made summaries of its surroundings. This
network instead reads the per-frame measurements as a sequence and learns its own summaries,
looking roughly 40 seconds either side of every frame. It follows the multi-stage design used
for action segmentation (MS-TCN): a first stage labels every frame, and later stages re-read
those labels and smooth them, which cuts the over-segmentation the ranking metric punishes.

Like the tree model it is trained only on other matches, and predicts twice -- as given and
with the teams swapped -- so it cannot favour one side.
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from . import model as tree_model
from .match import H, A, D

CONFIG = {"channels": 64, "layers": 10, "stages": 3, "dropout": 0.2}


def frame_tensor(match, f, names=None):
    """([n_frames, n_features] float32, names) of per-frame measurements."""
    base = tree_model.base_features(match, f, shape=True)
    names = names or sorted(base)
    return np.column_stack([base[name] for name in names]).astype(np.float32), names


class _Block(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.dilated = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.mix = nn.Conv1d(channels, channels, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop(self.mix(F.relu(self.dilated(x))))


class _Stage(nn.Module):
    def __init__(self, n_in, channels, layers, dropout, n_out=3):
        super().__init__()
        self.enter = nn.Conv1d(n_in, channels, 1)
        self.blocks = nn.ModuleList(_Block(channels, 2 ** i, dropout) for i in range(layers))
        self.leave = nn.Conv1d(channels, n_out, 1)

    def forward(self, x):
        x = self.enter(x)
        for block in self.blocks:
            x = block(x)
        return self.leave(x)


class Net(nn.Module):
    def __init__(self, n_features, channels=64, layers=10, stages=3, dropout=0.2):
        super().__init__()
        self.first = _Stage(n_features, channels, layers, dropout)
        self.refine = nn.ModuleList(_Stage(3, channels, layers, dropout)
                                    for _ in range(stages - 1))

    def forward(self, x):
        """x: [batch, n_features, time]. Returns the logits of every stage, last one final."""
        out = [self.first(x)]
        for stage in self.refine:
            out.append(stage(F.softmax(out[-1], dim=1)))
        return out


def boundary_weights(target, boost, radius=25):
    """Per-frame weight: `boost` within `radius` frames of a change of label, 1 elsewhere."""
    graded = target != 3
    change = torch.zeros_like(target, dtype=torch.float32)
    change[:, 1:] = ((target[:, 1:] != target[:, :-1]) & graded[:, 1:] & graded[:, :-1]).float()
    near = F.max_pool1d(change[:, None], 2 * radius + 1, stride=1, padding=radius)[:, 0]
    return (1.0 + (boost - 1.0) * near) * graded.float()


def loss_fn(stage_logits, target, smooth=0.15, boost=1.0):
    """Cross-entropy on graded frames plus a penalty on jumpy frame-to-frame predictions.

    With boost > 1 the frames around every change of possession or of dead/alive count more:
    those are the frames that decide whether a possession is found, and they are few.
    """
    weight = boundary_weights(target, boost) if boost != 1.0 else None
    total = 0.0
    for logits in stage_logits:
        if weight is None:
            total = total + F.cross_entropy(logits, target, ignore_index=3)
        else:
            per = F.cross_entropy(logits, target.clamp(max=2), reduction="none")
            total = total + (per * weight).sum() / weight.sum().clamp(min=1.0)
        log_prob = F.log_softmax(logits, dim=1)
        jump = (log_prob[:, :, 1:] - log_prob[:, :, :-1].detach()) ** 2
        total = total + smooth * torch.clamp(jump, max=16.0).mean()
    return total


def save(path, net, names, mean, std, config=CONFIG):
    torch.save({"state": net.state_dict(), "names": names, "mean": mean, "std": std,
                "config": config}, path)


def load_all(path, tags=None):
    """Every model in a checkpoint as [(net, ckpt)]. A checkpoint holds one model or several
    ("members") whose predictions are averaged; `tags` keeps only the members so tagged."""
    loaded = torch.load(path, map_location="cpu", weights_only=False)
    out = []
    for ckpt in loaded.get("members", [loaded]):
        if tags is not None and ckpt.get("tag") not in tags:
            continue
        net = Net(len(ckpt["names"]), **ckpt["config"])
        net.load_state_dict(ckpt["state"])
        net.eval()
        out.append((net, ckpt))
    return out


def load(path):
    return load_all(path)[0]


def predict_all(members, match, f, device="cpu"):
    """Mean of the members' probabilities."""
    total = None
    for net, ckpt in members:
        prob = predict(net, ckpt, match, f, device)
        total = prob if total is None else total + prob
    return total / len(members)


def _run(net, x, mean, std, device):
    scaled = (x - mean) / std
    with torch.no_grad():
        tensor = torch.from_numpy(scaled.T[None]).to(device)
        return F.softmax(net(tensor)[-1], dim=1)[0].T.cpu().numpy()


def predict(net, ckpt, match, f, device="cpu"):
    """[n_frames, 3] probabilities over H, A, D, averaged over both team orderings."""
    names, mean, std = ckpt["names"], ckpt["mean"], ckpt["std"]
    x, _ = frame_tensor(match, f, names)
    x_swapped, _ = frame_tensor(match, tree_model.swap_teams(f), names)
    prob = np.full((match.n_frames, 3), 1.0 / 3.0, dtype=np.float32)
    net = net.to(device)
    for start, stop in match.runs:
        p = _run(net, x[start:stop], mean, std, device)
        q = _run(net, x_swapped[start:stop], mean, std, device)
        prob[start:stop] = (p + q[:, [A, H, D]]) / 2.0
    return prob


def mirror(match):
    """The same match reflected across the halfway line's axis (top and bottom swapped).

    Football looks the same in a mirror, so a prediction made on the mirrored match is a second,
    slightly different opinion about the same frames; averaging the two cancels some errors.
    """
    from dataclasses import replace
    width = match.pitch[1]
    return replace(match, py=width - match.py, by=width - match.by)


def predict_mirrored(members, match, f=None, device="cpu"):
    """Mean of the members' probabilities on the match and on its mirror image."""
    from . import features
    f = f if f is not None else features.compute(match)
    flipped = mirror(match)
    return (predict_all(members, match, f, device)
            + predict_all(members, flipped, features.compute(flipped), device)) / 2.0


In [ ]:
%%writefile possession/shape.py
"""Team-shape measurements: what each team as a whole is doing on every frame.

The ball is unseen on about 41% of frames, in stretches of ten seconds and more. In those
stretches the only evidence of who has the ball is how the two teams are arranged and moving:
the team in possession spreads out and moves up the pitch, the other one closes up and drops.

Everything is measured in each team's own frame of reference -- distance from its own goal line
along its direction of attack -- so the two teams are described identically and swapping them
is just swapping the names. As elsewhere, nothing depends on which slot a player is in.
"""

import numpy as np

from .match import HOME, AWAY

LAG = 25          # frames over which movement is measured (1 s)
CHUNK = 10 ** 9   # one pass: movement is measured across frames, so no seams
SIDES = (("home", HOME), ("away", AWAY))


def attack_signs(match):
    """Per run, +1 if the home team attacks towards larger x, else -1.

    The goalkeeper gives it away: over a half, the team defending the goal at x = 0 has the
    player nearest to that goal line.
    """
    home = match.px[:, match.pteam == HOME]
    away = match.px[:, match.pteam == AWAY]
    signs = []
    for start, stop in match.runs:
        with np.errstate(all="ignore"):
            deepest_home = np.nanmedian(np.nanmin(home[start:stop], axis=1))
            deepest_away = np.nanmedian(np.nanmin(away[start:stop], axis=1))
        signs.append(1.0 if deepest_home <= deepest_away else -1.0)
    return signs


def _team(u, v, ball_u, ball_v, fps):
    """Shape of one team from its players' depth u and width v, in its own frame."""
    n = len(u)
    out = {}
    present = ~np.isnan(u)
    count = present.sum(axis=1)
    some = np.maximum(count, 1)
    with np.errstate(all="ignore"):
        cu = np.nansum(u, axis=1) / some
        cv = np.nansum(v, axis=1) / some
        su = np.sqrt(np.nansum((u - cu[:, None]) ** 2, axis=1) / some)
        sv = np.sqrt(np.nansum((v - cv[:, None]) ** 2, axis=1) / some)
    ordered = np.sort(u, axis=1)                      # NaN sorts last
    back = ordered[:, 1]                              # second deepest: the defensive line
    front = np.nanmax(np.where(present, u, -np.inf), axis=1)

    # Movement over the last second, using only the players tracked at both ends of it.
    du = np.full(u.shape, np.nan, dtype=np.float32)
    dv = np.full(u.shape, np.nan, dtype=np.float32)
    du[LAG:] = (u[LAG:] - u[:-LAG]) * (fps / float(LAG))
    dv[LAG:] = (v[LAG:] - v[:-LAG]) * (fps / float(LAG))
    jumped = np.hypot(du, dv) > 12.0
    du[jumped], dv[jumped] = np.nan, np.nan
    moving = np.maximum((~np.isnan(du)).sum(axis=1), 1)
    with np.errstate(all="ignore"):
        advance = np.nansum(du, axis=1) / moving
        pace = np.nansum(np.hypot(du, dv), axis=1) / moving
        fast = np.nansum(np.hypot(du, dv) > 4.0, axis=1) / moving
        back_move = np.zeros(n, dtype=np.float32)
        back_move[LAG:] = (back[LAG:] - back[:-LAG]) * (fps / float(LAG))

    dist = np.hypot(u - ball_u[:, None], v - ball_v[:, None])
    dist = np.sort(np.where(np.isnan(dist), np.inf, dist), axis=1)
    ahead = np.nansum(u > ball_u[:, None], axis=1) / some

    out["count"] = count
    out["depth"] = cu
    out["side"] = cv
    out["length"] = su
    out["width"] = sv
    out["line"] = back
    out["front"] = front
    out["advance"] = np.clip(advance, -8.0, 8.0)
    out["line_move"] = np.clip(back_move, -8.0, 8.0)
    out["pace"] = pace
    out["fast"] = fast
    out["ball_depth"] = ball_u
    out["ball_past_line"] = np.clip(ball_u - back, -40.0, 40.0)
    out["ahead_of_ball"] = ahead
    out["second_near"] = np.minimum(dist[:, 1], 40.0)
    out["third_near"] = np.minimum(dist[:, 2], 40.0)
    out["near_10m"] = (dist < 10.0).sum(axis=1)
    return out


def compute(match, bx, by):
    """{name_home / name_away: [n_frames] float32}. bx, by: the gap-filled ball."""
    length, width = match.pitch
    n = match.n_frames
    sign = np.ones(n, dtype=np.float32)
    for (start, stop), s in zip(match.runs, attack_signs(match)):
        sign[start:stop] = s
    ball_x = np.nan_to_num(bx, nan=length / 2.0)
    ball_y = np.nan_to_num(by, nan=width / 2.0)

    result = {}
    for name, team in SIDES:
        # The home team's frame follows `sign`; the away team attacks the other way.
        forward = sign if team == HOME else -sign
        flip = forward < 0
        cols = np.flatnonzero(match.pteam == team)
        parts = []
        for a in range(0, n, CHUNK):
            b = min(n, a + CHUNK)
            f = flip[a:b]
            u = np.where(f[:, None], length - match.px[a:b][:, cols], match.px[a:b][:, cols])
            v = np.where(f[:, None], width - match.py[a:b][:, cols], match.py[a:b][:, cols])
            ball_u = np.where(f, length - ball_x[a:b], ball_x[a:b])
            ball_v = np.where(f, width - ball_y[a:b], ball_y[a:b])
            parts.append(_team(u.astype(np.float32), v.astype(np.float32),
                               ball_u.astype(np.float32), ball_v.astype(np.float32), match.fps))
        for key in parts[0]:
            values = np.concatenate([p[key] for p in parts]).astype(np.float32)
            values[~np.isfinite(values)] = 0.0
            result["{}_{}".format(key, name)] = values
    return result


def swap(shape):
    """The same measurements with the two teams' names exchanged."""
    out = {}
    for key, values in shape.items():
        stem, side = key.rsplit("_", 1)
        out["{}_{}".format(stem, "away" if side == "home" else "home")] = values
    return out


In [ ]:
%%writefile possession/vendor/__init__.py
"""Third-party code kept unchanged: the organisers' metric library (MIT licence)."""


In [ ]:
%%writefile possession/vendor/possession_metrics.py
"""Metrics for per-frame possession inference, scored against a H/A/D/X ground truth.

numpy and pandas only, on purpose: no sklearn, nothing that pins a Python version, and
hand-rolling a 4x4 confusion matrix is fifteen lines. Keeping it dependency-free means the scorer
runs anywhere, and produces identical output under 3.8 and 3.11.

THE RANKING METRIC is spell_f1_mean -- possessions matched one-to-one by temporal IoU, defined
further down. macro-F1 over {H, A, D} is reported beside it as the per-frame view and is the first
tie-break. It is averaged unweighted over the frames where the ground truth is not X, rather than
counting frames, which is what stops a degenerate always-one-label submission from scoring
respectably, which is exactly what plain accuracy would do. It is however nearly blind to a uniform timing offset: lagging every
transition by ten frames barely moves it, because only a small fraction of frames sit near a
boundary. That is why the transition metrics below are computed as well, not as extras.

READ THIS BEFORE COMPARING TWO SCORES. A match has a great many frames but far fewer segments, so
consecutive frames are nowhere near independent and the effective sample size is the segment
count, not the frame count. One misjudged long dead-ball stretch moves macro-F1 by about a
percentage point on its own, so macro-F1 gaps below roughly 0.01 are segment-level noise rather
than signal. Set BOOTSTRAP_N to get a block-bootstrap interval over segments, which is the honest
way to answer "is this gap real"; do not compute per-frame intervals, they come out an order of
magnitude too narrow.

Usage:
    import possession_metrics as pm
    gt = pm.label_codes(pm.read_label_csv("gt/BAR-ATH_possession_gt.csv"))
    pred = pm.label_codes(pm.read_label_csv("results/mine.csv"))
    print(pm.evaluate(gt, pred)["macro_f1"])
"""

import numpy as np
import pandas as pd

# Params
# X is both "no ground truth here" (in the GT) and "I abstain" (in a submission). One symbol for
# both because the grading rule is the same either way: GT X is never scored, and a predicted X
# on a graded frame is a miss.
LABELS = ("H", "A", "D", "X")
SCORED = ("H", "A", "D")
H, A, D, X = 0, 1, 2, 3
N_LABELS = len(LABELS)
N_SCORED = len(SCORED)

# Tolerances for transition matching, in frames at 25 fps: ~0.5 s and 1.0 s. Reported as a pair
# because the gap between them is a lag detector -- a submission that scores badly at 12 and well
# at 25 has a systematic delay in that range, which no frame-level metric reveals.
TAUS = (12, 25)

# A possession is a team spell with the ball alive. D runs are stoppages, not possessions, so
# spells are maximal runs of H or of A. Dead-ball detection is covered separately by
# alive_dead_f1 and by the D row of the per-class table.
POSSESSION_LABELS = (H, A)
# IoU thresholds for possession-spell matching, averaged mAP-style. Averaging over IoU is the
# temporal action detection convention and is principled here, because IoU is a real quality
# axis for an interval. (Averaging over *frame tolerances* is not, and was rejected: the
# frame metric already costs only 0.0031 for a uniform 1-frame lag, below the noise floor.)
#
# Not called mAP anywhere on purpose: submissions are hard labels with no confidence scores, so
# there is no precision-recall curve to integrate. This is a mean F1 over IoU thresholds, and the
# mAP analogy is to the averaging axis alone.
SPELL_IOUS = (0.3, 0.4, 0.5, 0.6, 0.7)
SPELL_IOU_HEADLINE = 0.5
FPS = 25  # only used to report spell durations in seconds, which is how coaches read them

# Typical 5-95 percentile width of spell_f1_mean under a bootstrap over the ground-truth
# possessions of a full match, so a gap smaller than this is not a result. Measured across a set
# of reference submissions spanning realistic failure modes. It is 3-4x the ~0.01 floor on
# macro_f1, because the effective sample is the possessions scored hit-or-miss rather than every
# frame -- but the gaps between submissions widen in proportion, so exactly as many adjacent
# pairs stay separable under either metric.
SPELL_NOISE_BAND = 0.04

# Block bootstrap draws over GT segments. 0 = off (fast); 1000 for a final leaderboard.
BOOTSTRAP_N = 0
BOOTSTRAP_SEED = 0
BOOTSTRAP_PERCENTILES = (5, 95)

# Sentinel for possession share when a submission predicts no alive frames at all. Paired with
# share_defined in the output so it can never be mistaken for a measurement.
SHARE_UNDEFINED = 1.0


# -- reading and validation --------------------------------------------------------------------

def read_label_csv(path):
    """Read a frame,label CSV and return the label column as an object array.

    Tolerant about header case and surrounding whitespace, strict about everything else: the
    validation lives in validate_submission so a bad file can be reported rather than raised.
    """
    frame = pd.read_csv(path, dtype={"label": "object"})
    frame.columns = [str(c).strip().lower() for c in frame.columns]
    if "label" not in frame.columns or "frame" not in frame.columns:
        raise ValueError("expected columns 'frame' and 'label', got {}".format(
            list(frame.columns)))
    return frame


def validate_submission(frame, n_expected):
    """Return None if the table is a usable submission, else a one-line reason why it is not."""
    if len(frame) != n_expected:
        return "row count {} != {}".format(len(frame), n_expected)

    frames = frame["frame"].values
    if not np.issubdtype(frames.dtype, np.integer):
        return "'frame' column is not integer (dtype {})".format(frames.dtype)
    if not np.array_equal(frames, np.arange(n_expected)):
        # One comparison covers 0-based, monotonic, contiguous and duplicate-free.
        return "'frame' column is not 0..{} in order".format(n_expected - 1)

    labels = frame["label"].values
    blank = pd.isna(labels)
    if blank.any():
        # Never coerce a blank cell to X: an empty field is a mistake, while X is a deliberate
        # abstention, and scoring one as the other would hide the mistake.
        return "empty label in {} rows; write X to abstain".format(int(blank.sum()))

    cleaned = np.array([str(v).strip().upper() for v in labels], dtype=object)
    bad = sorted(set(cleaned) - set(LABELS))
    if bad:
        return "labels outside {}: {}".format(list(LABELS), bad[:5])
    return None


def label_codes(frame_or_labels):
    """Map label strings to 0..3. Accepts the DataFrame from read_label_csv or a bare array."""
    values = frame_or_labels
    if isinstance(values, pd.DataFrame):
        values = values["label"].values
    values = np.array([str(v).strip().upper() for v in values], dtype=object)

    codes = np.full(len(values), -1, dtype=np.int8)
    for index, name in enumerate(LABELS):
        codes[values == name] = index
    if (codes < 0).any():
        raise ValueError("labels outside {}: {}".format(
            list(LABELS), sorted(set(values[codes < 0]))))
    return codes


# -- frame-level metrics -----------------------------------------------------------------------

def confusion(gt_codes, pred_codes):
    """4x4 counts, rows = ground truth, cols = prediction."""
    pair = gt_codes.astype(np.int64) * N_LABELS + pred_codes.astype(np.int64)
    return np.bincount(pair, minlength=N_LABELS * N_LABELS).reshape(N_LABELS, N_LABELS)


def scored_confusion(cm):
    """The 3x4 slice everything is scored on: GT rows H/A/D only, all four prediction columns.

    The whole X policy is this slice. False positives are summed over three GT rows, so a frame
    the ground truth does not cover can never manufacture one. False negatives are summed over
    four prediction columns, so a predicted X on a graded frame does count as a miss. No
    special-casing anywhere else.
    """
    return cm[:N_SCORED, :]


def per_class_prf(cm3, k):
    """(precision, recall, f1) for class k from the 3x4 scored confusion."""
    true_positive = float(cm3[k, k])
    false_positive = float(cm3[:, k].sum()) - true_positive
    false_negative = float(cm3[k, :].sum()) - true_positive

    precision = true_positive / (true_positive + false_positive) \
        if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) \
        if true_positive + false_negative else 0.0
    if precision + recall == 0.0:
        return 0.0, 0.0, 0.0  # absent class or nothing predicted -> 0, never nan
    return precision, recall, 2.0 * precision * recall / (precision + recall)


def macro_f1(cm3):
    """Unweighted mean F1 over H, A and D. The metric the scoreboard ranks on."""
    return float(np.mean([per_class_prf(cm3, k)[2] for k in range(N_SCORED)]))


def accuracy(cm3):
    """Fraction of graded frames labelled correctly. Predicted X counts against."""
    total = float(cm3.sum())
    if not total:
        return 0.0
    return float(sum(cm3[k, k] for k in range(N_SCORED))) / total


def cohens_kappa(cm3):
    """Chance-corrected agreement. Exactly 0.0 for any constant-guess submission.

    Reported as a secondary column only. Its chance level is computed from the submission's own
    marginals, so shifting your predicted class proportions changes your score without improving
    any single decision -- a bad property to rank a leaderboard on.
    """
    square = np.zeros((N_LABELS, N_LABELS), dtype=np.float64)
    square[:N_SCORED, :] = cm3  # the unused GT X row stays zero and drops out of both terms
    total = square.sum()
    if not total:
        return 0.0

    observed = np.trace(square) / total
    expected = sum(square[k, :].sum() * square[:, k].sum()
                   for k in range(N_LABELS)) / total ** 2
    if expected >= 1.0:
        return 0.0
    return float((observed - expected) / (1.0 - expected))


def alive_dead_f1(cm3):
    """F1 of the alive/dead call alone, with ALIVE as the positive class.

    ALIVE and not DEAD because dead ball is a large share of all frames: with DEAD positive, an
    always-dead submission scores well on this and reads like competence. With ALIVE positive it
    scores 0, which is the honest answer.
    """
    true_positive = float(cm3[:D, :D].sum())              # GT alive, predicted alive
    false_positive = float(cm3[D, :D].sum())              # GT dead, predicted alive
    false_negative = float(cm3[:D, D:].sum())             # GT alive, predicted dead or X

    precision = true_positive / (true_positive + false_positive) \
        if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) \
        if true_positive + false_negative else 0.0
    if precision + recall == 0.0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def team_accuracy(cm3):
    """(strict, both_alive) H-vs-A accuracy.

    strict is over every GT-alive frame, so answering D or X there counts as wrong. both_alive is
    over the frames where GT and prediction agree the ball is alive, i.e. the team call in
    isolation, with the alive/dead call already charged by alive_dead_f1.
    """
    correct = float(cm3[H, H] + cm3[A, A])
    gt_alive = float(cm3[:D, :].sum())
    both_alive = float(cm3[:D, :D].sum())
    return (correct / gt_alive if gt_alive else 0.0,
            correct / both_alive if both_alive else 0.0)


def possession_share(cm3):
    """(gt_share, pred_share, abs_err, defined) home share of alive time.

    Each side is measured over its own alive frames, because this is the number a submission
    would actually report as "the home side had 55% of the ball" -- not a per-frame agreement.
    """
    gt_alive = float(cm3[:D, :].sum())
    pred_alive = float(cm3[:, :D].sum())
    gt_share = float(cm3[H, :].sum()) / gt_alive if gt_alive else 0.0
    if not pred_alive:
        return gt_share, 0.0, SHARE_UNDEFINED, False
    pred_share = float(cm3[:, H].sum()) / pred_alive
    return gt_share, pred_share, abs(pred_share - gt_share), True


# -- temporal structure ------------------------------------------------------------------------

def valid_runs(gt_codes):
    """Half-open [start, stop) spans where the ground truth is not X.

    Derived from the data, never hard-coded: for BAR-ATH this returns the two halves,
    (11185, 90652) and (110808, 190198). Everything temporal is computed inside a run and never
    across one, so the 20156-frame halftime hole cannot produce a phantom transition. That
    matters even when the labels either side happen to match -- here they are both D, so a
    whole-array run-length encoding silently merges them and undercounts the segments by one.
    """
    graded = (gt_codes != X).astype(np.int8)
    edges = np.diff(np.concatenate(([0], graded, [0])))
    return list(zip(np.flatnonzero(edges == 1).tolist(),
                    np.flatnonzero(edges == -1).tolist()))


def fill_abstentions(pred_codes, runs):
    """Copy of pred_codes with X forward-filled (then back-filled at the run head) inside runs.

    Frame-level metrics see the raw X and charge it. The temporal metrics see this filled version
    instead, because 'H X X H' is one uninterrupted possession that the model went quiet during,
    not two possessions with a pair of transitions in between. Without this, an abstention is
    charged twice: once as a frame-level miss and again as two spurious transitions.
    """
    filled = pred_codes.copy()
    for start, stop in runs:
        window = filled[start:stop]
        known = np.flatnonzero(window != X)
        if not len(known):
            continue  # a run the submission abstained on entirely; nothing to carry forward
        # For each position, the index of the most recent non-X at or before it, then clamp the
        # leading positions to the first known value (the back-fill).
        carry = np.maximum.accumulate(np.where(window != X, np.arange(len(window)), -1))
        carry[carry < 0] = known[0]
        filled[start:stop] = window[carry]
    return filled


def segments(codes, runs):
    """[(start, stop, code)] constant-label runs, encoded inside each valid run separately."""
    out = []
    for start, stop in runs:
        window = codes[start:stop]
        if not len(window):
            continue
        breaks = np.flatnonzero(window[1:] != window[:-1]) + 1
        bounds = np.concatenate(([0], breaks, [len(window)]))
        for i in range(len(bounds) - 1):
            out.append((start + int(bounds[i]), start + int(bounds[i + 1]),
                        int(window[bounds[i]])))
    return out


def transitions(codes, runs):
    """[(frame, from_code, to_code)] label changes, never spanning a run boundary.

    frame is the first frame of the new label, so it is directly usable as a seek target in a
    video player.
    """
    out = []
    for start, stop in runs:
        window = codes[start:stop]
        for index in (np.flatnonzero(window[1:] != window[:-1]) + 1).tolist():
            out.append((start + int(index), int(window[index - 1]), int(window[index])))
    return out


def match_transitions(gt_trans, pred_trans, tol, typed="to"):
    """One-to-one match of predicted transitions to ground-truth ones within +/- tol frames.

    Returns (pairs, missed, spurious), pairs being [(gt_transition, pred_transition)].

    Matching is on the destination label by default. 'D -> H' and 'D -> A' at the same frame are
    materially different events -- you spotted the restart but gave it to the wrong team -- so the
    destination has to agree. The origin does not: a predicted 'A -> H' against a true 'D -> H'
    got the event right and only the preceding state wrong, and that preceding state is already
    fully charged at frame level. typed="pair" demands both, typed="none" neither.

    Both lists are frame-sorted and feasibility is an interval, so matching earliest-to-earliest
    within each key is maximum-cardinality (the standard interval exchange argument) as well as
    deterministic. Cross-run pairing needs no explicit guard: runs are 20156 frames apart and
    tol is at most a few dozen, so the interval test rejects it.

    Caveat worth knowing: maximum cardinality does not mean minimum total latency, and when
    several maximum matchings exist the reported mean latency can shift by a few frames. A
    handful of ground-truth segments are shorter than the loose tolerance, so at tol=25 the
    windows really can overlap. That is what n_ambiguous in transition_scores is for.
    """
    def key(transition):
        if typed == "pair":
            return (transition[1], transition[2])
        if typed == "none":
            return 0
        return transition[2]

    pairs, missed, spurious = [], [], []
    gt_by_key, pred_by_key = {}, {}
    for transition in gt_trans:
        gt_by_key.setdefault(key(transition), []).append(transition)
    for transition in pred_trans:
        pred_by_key.setdefault(key(transition), []).append(transition)

    for group in sorted(set(gt_by_key) | set(pred_by_key)):
        gts = gt_by_key.get(group, [])
        preds = pred_by_key.get(group, [])
        i = j = 0
        while i < len(gts) and j < len(preds):
            delta = preds[j][0] - gts[i][0]
            if delta < -tol:
                spurious.append(preds[j])  # too early for this or any later gt
                j += 1
            elif delta > tol:
                missed.append(gts[i])      # no remaining pred can reach back to it
                i += 1
            else:
                pairs.append((gts[i], preds[j]))
                i += 1
                j += 1
        missed.extend(gts[i:])
        spurious.extend(preds[j:])
    return pairs, missed, spurious


def transition_scores(gt_trans, pred_trans, tol, typed="to"):
    """Precision, recall, F1 and latency for transition detection at one tolerance.

    Precision and recall are reported separately and always should be: jitter shows up as recall
    near 1.0 with precision near 0, which a single F1 hides.
    """
    pairs, missed, spurious = match_transitions(gt_trans, pred_trans, tol, typed)
    true_positive, false_positive, false_negative = len(pairs), len(spurious), len(missed)

    precision = true_positive / float(true_positive + false_positive) \
        if true_positive + false_positive else 0.0
    recall = true_positive / float(true_positive + false_negative) \
        if true_positive + false_negative else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0

    latencies = np.array([pred[0] - gt[0] for gt, pred in pairs], dtype=np.float64)
    ambiguous = _count_ambiguous(gt_trans, pred_trans, tol, typed)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_gt": len(gt_trans),
        "n_pred": len(pred_trans),
        "n_matched": true_positive,
        "n_missed": false_negative,
        "n_spurious": false_positive,
        "n_ambiguous": ambiguous,
        "mean_latency": float(latencies.mean()) if len(latencies) else None,
        "median_latency": float(np.median(latencies)) if len(latencies) else None,
        "mean_abs_latency": float(np.abs(latencies).mean()) if len(latencies) else None,
        "p90_abs_latency": float(np.percentile(np.abs(latencies), 90)) if len(latencies) else None,
        "pairs": pairs,
        "missed": missed,
        "spurious": spurious,
    }


def _count_ambiguous(gt_trans, pred_trans, tol, typed):
    """GT transitions with more than one feasible prediction, i.e. where the match was a choice.

    When this is 0 the latency figures are exact; when it is large they are indicative only.
    """
    def key(transition):
        if typed == "pair":
            return (transition[1], transition[2])
        if typed == "none":
            return 0
        return transition[2]

    by_key = {}
    for transition in pred_trans:
        by_key.setdefault(key(transition), []).append(transition[0])
    for group in by_key:
        by_key[group] = np.array(sorted(by_key[group]))

    count = 0
    for transition in gt_trans:
        frames = by_key.get(key(transition))
        if frames is None:
            continue
        if int(np.sum(np.abs(frames - transition[0]) <= tol)) > 1:
            count += 1
    return count


def fragmentation(gt_codes, pred_codes, runs):
    """(n_gt_segments, n_pred_segments, ratio). A jitter detector: a flickering submission can
    hold a respectable macro-F1 while producing thirty times too many possessions."""
    n_gt = len(segments(gt_codes, runs))
    n_pred = len(segments(pred_codes, runs))
    return n_gt, n_pred, (n_pred / float(n_gt) if n_gt else 0.0)


# -- possession-spell level --------------------------------------------------------------------
#
# Why this level exists at all. Frame macro-F1 is the right per-frame metric, but no per-frame
# metric can represent possession, because the overwhelming majority of graded frames sit in the
# interior of a segment where nothing is happening. A jittery submission can hold a per-frame
# score near 0.97 while reporting well over half as many possessions again as the truth, each far
# too short. Per-frame scoring also over-weights dead ball, which is a much larger share of frames
# than of segments, and lets the longest possession of a match count hundreds of times the
# shortest. Treating each possession as one object fixes all three.

def possession_spells(codes, runs):
    """[(start, stop, code)] maximal H or A runs -- the possessions, stoppages excluded.

    Built on the gap-aware segments(), so no spell can ever straddle the halftime hole.
    """
    return [(a, b, k) for a, b, k in segments(codes, runs) if k in POSSESSION_LABELS]


def spell_overlaps(gt_spells, pred_spells):
    """[(iou, gt_index, pred_index)] for every same-team overlapping pair, sorted for matching.

    Computed once and filtered per threshold by match_spells: the IoU of a pair does not depend
    on the threshold, so recomputing it for each of SPELL_IOUS would be five times the work for
    identical results. Sorted by (-iou, gt_index, pred_index) so ties break the same way on
    every run and the matching is reproducible.
    """
    out = []
    for i, (gt_start, gt_stop, gt_code) in enumerate(gt_spells):
        for j, (pred_start, pred_stop, pred_code) in enumerate(pred_spells):
            if pred_code != gt_code or pred_stop <= gt_start or gt_stop <= pred_start:
                continue
            intersection = min(gt_stop, pred_stop) - max(gt_start, pred_start)
            union = max(gt_stop, pred_stop) - min(gt_start, pred_start)
            out.append((intersection / float(union), i, j))
    out.sort(key=lambda item: (-item[0], item[1], item[2]))
    return out


def matched_gt_flags(overlaps, n_gt, threshold):
    """Boolean array over ground-truth spells, True where one was matched at this IoU.

    Greedy on the IoU-descending list, marking both sides consumed, so every ground-truth spell
    is credited to at most one prediction and vice versa. Returned per spell rather than as a
    count because the bootstrap needs to resample individual possessions.
    """
    flags = np.zeros(n_gt, dtype=bool)
    matched_pred = set()
    for iou, gt_index, pred_index in overlaps:
        if iou < threshold:
            break  # the list is IoU-descending, so nothing later can qualify either
        if flags[gt_index] or pred_index in matched_pred:
            continue
        flags[gt_index] = True
        matched_pred.add(pred_index)
    return flags


def match_spells(overlaps, n_gt, n_pred, threshold):
    """(tp, fp, fn) from a one-to-one greedy match of spells at one IoU threshold."""
    true_positive = int(matched_gt_flags(overlaps, n_gt, threshold).sum())
    return true_positive, n_pred - true_positive, n_gt - true_positive


def spell_scores(gt_codes, pred_codes, runs, ious=SPELL_IOUS):
    """Possession-spell detection precision/recall/F1 at each IoU, plus the mean F1 over them."""
    gt_spells = possession_spells(gt_codes, runs)
    pred_spells = possession_spells(pred_codes, runs)
    overlaps = spell_overlaps(gt_spells, pred_spells)

    per_iou, f1s = {}, []
    for threshold in ious:
        true_positive, false_positive, false_negative = match_spells(
            overlaps, len(gt_spells), len(pred_spells), threshold)
        precision = true_positive / float(true_positive + false_positive) \
            if true_positive + false_positive else 0.0
        recall = true_positive / float(true_positive + false_negative) \
            if true_positive + false_negative else 0.0
        f1 = 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_iou[threshold] = {"precision": precision, "recall": recall, "f1": f1,
                              "n_matched": true_positive, "n_spurious": false_positive,
                              "n_missed": false_negative}
        f1s.append(f1)

    return {
        "per_iou": per_iou,
        "mean_f1": float(np.mean(f1s)) if f1s else 0.0,
        "n_gt": len(gt_spells),
        "n_pred": len(pred_spells),
    }


def possession_stats(codes, runs):
    """What a coach reads off a possession signal: how many, how long, and the home share.

    Reported alongside the F1s because they need no metric literacy -- a predicted possession
    count next to the truth's states the failure more plainly than any score. Note the home share
    is a poor discriminator on its own: a jittery submission can get it almost exactly right while
    inventing hundreds of possessions, so it is reported and never ranked on.
    """
    spells = possession_spells(codes, runs)
    lengths = [b - a for a, b, _ in spells]
    alive = int(np.sum((codes == H) | (codes == A)))
    return {
        "n_spells": len(spells),
        "mean_duration_s": float(np.mean(lengths)) / FPS if lengths else 0.0,
        "home_share": float(np.sum(codes == H)) / alive if alive else 0.0,
    }


# -- bootstrap ---------------------------------------------------------------------------------

def bootstrap_macro_f1(gt_codes, pred_codes, runs, n_draws,
                       seed=BOOTSTRAP_SEED, percentiles=BOOTSTRAP_PERCENTILES):
    """Block-bootstrap interval for macro-F1, resampling whole GT segments.

    Segments, not frames, because frames inside a possession are not independent observations.
    Returns (low, high) or (None, None) when disabled.
    """
    if not n_draws:
        return None, None

    gt_segments = segments(gt_codes, runs)
    if not gt_segments:
        return None, None

    # Per-segment prediction histogram, so a draw is a matrix add rather than a rescan of 199k
    # frames: contributions[i, p] = frames of segment i the submission called p.
    gt_of_segment = np.array([code for _, _, code in gt_segments], dtype=np.int64)
    contributions = np.zeros((len(gt_segments), N_LABELS), dtype=np.int64)
    for index, (start, stop, _) in enumerate(gt_segments):
        contributions[index] = np.bincount(pred_codes[start:stop].astype(np.int64),
                                           minlength=N_LABELS)

    rng = np.random.RandomState(seed)
    scores = np.empty(n_draws, dtype=np.float64)
    for draw in range(n_draws):
        picks = rng.randint(0, len(gt_segments), len(gt_segments))
        cm3 = np.zeros((N_SCORED, N_LABELS), dtype=np.int64)
        np.add.at(cm3, gt_of_segment[picks], contributions[picks])
        scores[draw] = macro_f1(cm3)
    low, high = np.percentile(scores, percentiles)
    return float(low), float(high)


def bootstrap_spell_f1(gt_codes, pred_codes, runs, n_draws,
                       seed=BOOTSTRAP_SEED, percentiles=BOOTSTRAP_PERCENTILES):
    """Bootstrap interval for spell_f1_mean, resampling whole possessions.

    Possessions, not segments and not frames, because spell_f1_mean scores each possession as one
    hit-or-miss observation, and that count is the sample size governing its variance. This is the
    interval that matters once the board ranks on spellF1; bootstrap_macro_f1 above answers the
    same question for the frame-level metric.

    Returns (low, high) or (None, None) when disabled.
    """
    if not n_draws:
        return None, None

    gt_spells = possession_spells(gt_codes, runs)
    if not gt_spells:
        return None, None
    pred_spells = possession_spells(pred_codes, runs)
    overlaps = spell_overlaps(gt_spells, pred_spells)
    n_gt, n_pred = len(gt_spells), len(pred_spells)

    # Matched-or-not per possession per threshold, computed once. A draw is then a resample of
    # rows rather than a re-run of the matching.
    flags = np.vstack([matched_gt_flags(overlaps, n_gt, threshold)
                       for threshold in SPELL_IOUS])

    rng = np.random.RandomState(seed)
    scores = np.empty(n_draws, dtype=np.float64)
    for draw in range(n_draws):
        picks = rng.randint(0, n_gt, n_gt)
        f1s = []
        for row in flags:
            true_positive = int(row[picks].sum())
            false_negative = n_gt - true_positive
            # n_pred is fixed while the resampled tp moves, so for a submission predicting fewer
            # spells than the truth a draw can imply tp > n_pred. Clamp rather than go negative.
            false_positive = max(0, n_pred - true_positive)
            f1s.append(2.0 * true_positive / (2 * true_positive + false_positive + false_negative)
                       if true_positive else 0.0)
        scores[draw] = np.mean(f1s)
    low, high = np.percentile(scores, percentiles)
    return float(low), float(high)


# -- entry point -------------------------------------------------------------------------------

def evaluate(gt_codes, pred_codes, taus=TAUS, bootstrap_n=BOOTSTRAP_N, typed="to"):
    """Every metric for one submission. The only function run_evaluation.py needs.

    Touches the full-length arrays exactly twice: once for the confusion matrix, once for the
    run-length encoding behind the temporal metrics.
    """
    if len(gt_codes) != len(pred_codes):
        raise ValueError("length mismatch: gt {} vs pred {}".format(
            len(gt_codes), len(pred_codes)))

    cm = confusion(gt_codes, pred_codes)
    cm3 = scored_confusion(cm)

    runs = valid_runs(gt_codes)
    filled = fill_abstentions(pred_codes, runs)
    gt_trans = transitions(gt_codes, runs)
    pred_trans = transitions(filled, runs)

    n_gt_seg, n_pred_seg, frag = fragmentation(gt_codes, filled, runs)
    gt_share, pred_share, share_err, share_defined = possession_share(cm3)
    strict_team, both_alive_team = team_accuracy(cm3)
    boot_low, boot_high = bootstrap_macro_f1(gt_codes, pred_codes, runs, bootstrap_n)
    spell_boot_low, spell_boot_high = bootstrap_spell_f1(gt_codes, filled, runs, bootstrap_n)

    # Possession-spell level, on the abstention-filled prediction for the same reason the
    # transition metrics use it: 'H X X H' is one possession the model went quiet during, not
    # two possessions with a gap.
    spells = spell_scores(gt_codes, filled, runs)
    gt_stats = possession_stats(gt_codes, runs)
    pred_stats = possession_stats(filled, runs)

    result = {
        "macro_f1": macro_f1(cm3),
        # Co-headline, deliberately not folded into macro_f1: they answer different questions
        # ("right label" vs "right possessions") and their disagreement is the useful signal.
        "spell_f1_mean": spells["mean_f1"],
        "spell_f1_at_headline": spells["per_iou"][SPELL_IOU_HEADLINE]["f1"],
        "spell_iou_headline": SPELL_IOU_HEADLINE,
        "spell_bootstrap_low": spell_boot_low,
        "spell_bootstrap_high": spell_boot_high,
        "n_gt_spells": spells["n_gt"],
        "n_pred_spells": spells["n_pred"],
        "spell_count_error": spells["n_pred"] - spells["n_gt"],
        "mean_spell_duration_s": pred_stats["mean_duration_s"],
        "gt_mean_spell_duration_s": gt_stats["mean_duration_s"],
        "accuracy": accuracy(cm3),
        "kappa": cohens_kappa(cm3),
        "alive_dead_f1": alive_dead_f1(cm3),
        "team_acc_gt_alive": strict_team,
        "team_acc_both_alive": both_alive_team,
        "gt_home_share": gt_share,
        "pred_home_share": pred_share,
        "share_err": share_err,
        "share_defined": share_defined,
        "n_gt_segments": n_gt_seg,
        "n_pred_segments": n_pred_seg,
        "fragmentation": frag,
        "n_frames": int(len(gt_codes)),
        "n_graded": int(cm3.sum()),
        "n_abstain_on_graded": int(cm3[:, X].sum()),
        "bootstrap_low": boot_low,
        "bootstrap_high": boot_high,
        "bootstrap_n": bootstrap_n,
        "confusion": cm.tolist(),
        "per_class": {},
        "transitions": {},
        # The per-IoU breakdown is for the detail block and the JSON report, not the CSV: five
        # thresholds x three figures would add fifteen columns for what the mean already ranks.
        "spell_f1": spells["per_iou"],
        "gt_home_share_spells": gt_stats["home_share"],
    }
    for k, name in enumerate(SCORED):
        precision, recall, f1 = per_class_prf(cm3, k)
        result["per_class"][name] = {
            "precision": precision, "recall": recall, "f1": f1,
            "support": int(cm3[k, :].sum()),
        }
    for tol in taus:
        result["transitions"][tol] = transition_scores(gt_trans, pred_trans, tol, typed)
    return result


In [ ]:
%%writefile scripts/download_external.py
"""Download the seven open DFL / Sportec matches (other games, CC-BY 4.0) into external/dfl.

Source: the Hugging Face mirror of Bassek et al., "An integrated dataset of spatiotemporal and
event data in elite soccer", Scientific Data 2025. About 2.6 GB.

    python scripts/download_external.py
"""

import os
import sys
import urllib.request

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
TARGET = os.path.join(ROOT, "external", "dfl")
BASE = "https://huggingface.co/datasets/pysport/idsse-data/resolve/main/"
MATCHES = ("000001_DFL-MAT-J03WMX", "000001_DFL-MAT-J03WN1", "000002_DFL-MAT-J03WOH",
           "000002_DFL-MAT-J03WOY", "000002_DFL-MAT-J03WPY", "000002_DFL-MAT-J03WQQ",
           "000002_DFL-MAT-J03WR9")
KINDS = ("DFL_02_01_matchinformation", "DFL_04_03_positions_raw_observed")


if __name__ == '__main__':
    os.makedirs(TARGET, exist_ok=True)
    for match in MATCHES:
        for kind in KINDS:
            name = "{}_DFL-COM-{}.xml".format(kind, match)
            path = os.path.join(TARGET, name)
            if os.path.isfile(path) and os.path.getsize(path) > 0:
                continue
            print("downloading {}".format(name), flush=True)
            partial = path + ".part"
            urllib.request.urlretrieve(BASE + name, partial)
            os.replace(partial, path)
    print("done: {}".format(TARGET))
    sys.exit(0)


In [ ]:
%%writefile scripts/prepare_external.py
"""Convert the downloaded open DFL matches into cached Match files and describe them.

    python scripts/prepare_external.py            # all matches in external/dfl
"""

import os
import sys

import numpy as np

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
from possession import io_dfl  # noqa: E402
from possession.match import Match, H, A, D  # noqa: E402
from possession.vendor import possession_metrics as pm  # noqa: E402
from possession.scoring import graded_truth  # noqa: E402

DFL_DIR = os.path.join(ROOT, "external", "dfl")
CACHE = os.path.join(ROOT, "cache")


def describe(match):
    truth = graded_truth(match)
    runs = pm.valid_runs(truth)
    stats = pm.possession_stats(truth, runs)
    graded = truth[truth != 3]
    minutes = len(graded) / match.fps / 60.0
    shares = [float(np.mean(graded == k)) for k in (H, A, D)]

    # Does BallPossession==1 really mean the home side? Nearest player to a live ball should
    # agree with the label far more often than not.
    alive = np.flatnonzero((truth == H) | (truth == A))[::10]
    dist = np.hypot(match.px[alive] - match.bx[alive, None], match.py[alive] - match.by[alive, None])
    dist = np.where(np.isnan(dist), np.inf, dist)
    nearest_team = match.pteam[np.argmin(dist, axis=1)]
    agree = float(np.mean(nearest_team == truth[alive]))

    print("{:<16} {:6.1f} min  H {:.3f} A {:.3f} D {:.3f}  possessions {:>4} ({:.2f}/min) "
          "mean {:5.1f} s  nearest-player agreement {:.3f}  slots {}".format(
              match.name, minutes, shares[0], shares[1], shares[2], stats["n_spells"],
              stats["n_spells"] / minutes, stats["mean_duration_s"], agree, match.px.shape[1]))


if __name__ == '__main__':
    os.makedirs(CACHE, exist_ok=True)
    for match_id in io_dfl.match_ids(DFL_DIR):
        path = os.path.join(CACHE, "DFL-{}.npz".format(match_id))
        if os.path.isfile(path):
            match = Match.load(path)
        else:
            match = io_dfl.load(DFL_DIR, match_id)
            match.save(path)
        describe(match)


In [ ]:
%%writefile scripts/download_metrica.py
"""Download Metrica Sports' three public sample games (other games, anonymised) into external/metrica.

Source: https://github.com/metrica-sports/sample-data -- tracking (25 fps, players and ball)
and synchronised event data. About 210 MB. Used only as extra training and test matches.

    python scripts/download_metrica.py
"""

import os
import sys
import urllib.request

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
TARGET = os.path.join(ROOT, "external", "metrica")
BASE = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/"
FILES = (
    "data/Sample_Game_1/Sample_Game_1_RawEventsData.csv",
    "data/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv",
    "data/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv",
    "data/Sample_Game_2/Sample_Game_2_RawEventsData.csv",
    "data/Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv",
    "data/Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv",
    "data/Sample_Game_3/Sample_Game_3_events.json",
    "data/Sample_Game_3/Sample_Game_3_metadata.xml",
    "data/Sample_Game_3/Sample_Game_3_tracking.txt",
    "documentation/events-definitions.pdf",
)


if __name__ == '__main__':
    os.makedirs(TARGET, exist_ok=True)
    for name in FILES:
        path = os.path.join(TARGET, os.path.basename(name))
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            continue
        print("downloading {}".format(os.path.basename(name)), flush=True)
        partial = path + ".part"
        urllib.request.urlretrieve(BASE + name, partial)
        os.replace(partial, path)
    print("done: {}".format(TARGET))
    sys.exit(0)


In [ ]:
%%writefile scripts/prepare_metrica.py
"""Convert the Metrica sample games in external/metrica to cache/MET-*.npz and describe them.

The description is printed next to the same numbers for a DFL match, as a check that labels
built from events look like labels built from tracking: number and length of possessions,
share of dead ball, and how often the team in possession is the one nearest the ball.

    python scripts/prepare_metrica.py
"""

import glob
import os
import sys
import warnings

import numpy as np

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
from possession import io_metrica, relabel  # noqa: E402
from possession.match import Match, H, A, D  # noqa: E402
from possession.scoring import graded_truth  # noqa: E402
from possession.vendor import possession_metrics as pm  # noqa: E402

SOURCE = os.path.join(ROOT, "external", "metrica")
CACHE = os.path.join(ROOT, "cache")
MIN_CONTROL = 50


def describe(match):
    labels = relabel.controlled(match.labels, match.runs, MIN_CONTROL)
    from dataclasses import replace
    truth = graded_truth(replace(match, labels=labels))
    raw = pm.possession_stats(match.labels, match.runs)
    stats = pm.possession_stats(truth, match.runs)
    rows = np.concatenate([np.arange(a, b) for a, b in match.runs])
    lab = truth[rows]
    alive = (lab == H) | (lab == A)
    # Who is nearest the ball, where someone is within 1 m of it.
    home, away = match.pteam == 0, match.pteam == 1
    d = np.hypot(match.px[rows] - match.bx[rows, None], match.py[rows] - match.by[rows, None])
    d = np.where(np.isnan(d), np.inf, d)
    dh, da = d[:, home].min(axis=1), d[:, away].min(axis=1)
    close = alive & (np.minimum(dh, da) < 1.0)
    nearest = np.where(dh < da, H, A)
    agree = float((nearest[close] == lab[close]).mean()) if close.any() else float("nan")
    seen = ~np.isnan(match.bx[rows])
    moving = np.hypot(np.diff(match.bx[rows]), np.diff(match.by[rows])) * match.fps
    dead = lab[1:] == D
    return ("{:<12} minutes {:5.1f}  possessions raw {:4d} -> controlled {:4d}  mean {:5.1f} s  "
            "dead share {:.3f}  ball seen {:.3f} (alive {:.3f}, dead {:.3f})  "
            "nearest team agrees {:.3f}  ball still while dead {:.3f}".format(
                match.name, len(rows) / match.fps / 60.0, raw["n_spells"], stats["n_spells"],
                stats["mean_duration_s"], float((lab == D).mean()), float(seen.mean()),
                float(seen[alive].mean()), float(seen[lab == D].mean()) if (lab == D).any() else 0.0,
                agree, float(np.nanmean(moving[dead] < 0.1)) if dead.any() else 0.0))


if __name__ == '__main__':
    os.makedirs(CACHE, exist_ok=True)
    for number in (1, 2, 3):
        path = os.path.join(CACHE, "MET-{}.npz".format(number))
        match = Match.load(path) if os.path.isfile(path) else io_metrica.load(SOURCE, number)
        if not os.path.isfile(path):
            match.save(path)
        print(describe(match), flush=True)
    dfl = sorted(glob.glob(os.path.join(CACHE, "DFL-*.npz")))
    for path in dfl[:2]:
        print(describe(Match.load(path)), flush=True)


In [ ]:
%%writefile scripts/train_seq.py
"""Train the sequence model on the external matches. Meant for a GPU (e.g. a Kaggle notebook).

Leave-one-match-out first, so the score printed is from matches the model never saw and can be
compared with the tree model's; then one final model trained on every match.

    python scripts/train_seq.py                                  # full run, GPU recommended
    python scripts/train_seq.py --copies 1 --cv-steps 20 --steps 20 --dfl 1 --metrica 1 --no-stress   # smoke test
"""

import argparse
import glob
import os
import sys
import time
import warnings
from multiprocessing import Pool

import numpy as np

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
import torch  # noqa: E402

from possession import decode, features, model as tree_model, relabel, seqmodel  # noqa: E402
from possession.degrade import Noise, degrade, random_noise  # noqa: E402
from possession.match import Match, H, A, D  # noqa: E402
from possession.scoring import graded_truth, line, score  # noqa: E402

MIN_CONTROL = 50
CROP = 4096
BATCH = 8
SWAP = np.array([A, H, D, 3], dtype=np.int64)
WIDE = {"channels": 96, "layers": 11, "stages": 3, "dropout": 0.2}
DECODERS = ((2.0, 2.0), (4.0, 8.0), (8.0, 16.0), (12.0, 32.0), (16.0, 48.0))
# Held-out matches are scored under the calibrated noise and under a few others, to see how
# much the score depends on the noise guess being right.
STRESS = {
    "calibrated": Noise(),
    "ball unseen 55%": Noise(ball_missing=0.55),
    "short gaps, jitter": Noise(ball_mean_gap=40.0, ball_fill=0, ball_smooth=1, ball_jitter=0.15,
                                ball_drift=0.0, player_mean_gap=30.0, player_fill=0,
                                player_smooth=1, player_jitter=0.10, dead_hidden=0.25,
                                cam_height=0.0, air_lost=99.0, out_seen=1.0),
}


def prepare(job):
    """One degraded copy of one match: frame tensors for both team orderings, labels, runs."""
    path, seed, variant = job
    default_noise = variant is not None
    match = Match.load(path)
    match.labels = relabel.controlled(match.labels, match.runs, MIN_CONTROL)
    truth = graded_truth(match)
    noise = STRESS[variant] if default_noise else random_noise(np.random.default_rng(1000 + seed))
    noisy = degrade(match, noise, seed=seed)
    f = features.compute(noisy)
    x, names = seqmodel.frame_tensor(noisy, f)
    x_swapped, _ = seqmodel.frame_tensor(noisy, tree_model.swap_teams(f), names)
    return {"x": x.astype(np.float16), "x_swapped": x_swapped.astype(np.float16),
            "y": truth.astype(np.int64), "runs": noisy.runs,
            "names": names, "match": noisy if default_noise else None,
            "features": f if default_noise else None}


def batches(copies, mean, std, rng, device, cols):
    """Endless random crops, half of them with the teams swapped."""
    while True:
        xs, ys = [], []
        for _ in range(BATCH):
            item = copies[rng.integers(len(copies))]
            start, stop = item["runs"][rng.integers(len(item["runs"]))]
            a = int(rng.integers(start, max(start + 1, stop - CROP)))
            b = min(stop, a + CROP)
            swap = rng.random() < 0.5
            x = (item["x_swapped" if swap else "x"][a:b][:, cols].astype(np.float32) - mean) / std
            y = item["y"][a:b]
            if swap:
                y = SWAP[y]
            if b - a < CROP:  # a short half: pad, with padded frames ungraded
                x = np.vstack([x, np.zeros((CROP - (b - a), x.shape[1]), dtype=np.float32)])
                y = np.concatenate([y, np.full(CROP - (b - a), 3, dtype=np.int64)])
            xs.append(x.T)
            ys.append(y)
        yield (torch.from_numpy(np.stack(xs)).to(device),
               torch.from_numpy(np.stack(ys)).to(device))


def train(copies, steps, device, seed, label, config=None, cols=None, smooth=0.15, boost=1.0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    cols = np.arange(copies[0]["x"].shape[1]) if cols is None else np.asarray(cols)
    sample = np.vstack([c["x"][::50][:, cols].astype(np.float32) for c in copies])
    mean = sample.mean(axis=0).astype(np.float32)
    std = (sample.std(axis=0) + 1e-3).astype(np.float32)

    net = seqmodel.Net(len(mean), **(config or seqmodel.CONFIG)).to(device)
    optimiser = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=1e-3, total_steps=steps)
    net.train()
    started, running = time.time(), 0.0
    for step, (x, y) in enumerate(batches(copies, mean, std, rng, device, cols), start=1):
        loss = seqmodel.loss_fn(net(x), y, smooth=smooth, boost=boost)
        optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimiser.step()
        schedule.step()
        running += float(loss)
        if step % max(1, steps // 10) == 0 or step == steps:
            every = max(1, steps // 10)
            print("  {} step {}/{}  loss {:.4f}  ({:.0f} s)".format(
                label, step, steps, running / every, time.time() - started), flush=True)
            running = 0.0
        if step >= steps:
            break
    net.eval()
    return net, mean, std


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument("--copies", type=int, default=6, help="degraded copies per match")
    parser.add_argument("--steps", type=int, default=6000, help="steps for the final model")
    parser.add_argument("--cv-steps", type=int, default=3000, help="steps per held-out test")
    parser.add_argument("--dfl", type=int, default=0, help="use only the first N DFL matches (testing)")
    parser.add_argument("--metrica", type=int, default=3, help="Metrica matches to use")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--no-stress", action="store_true", help="calibrated noise only")
    parser.add_argument("--seeds", type=int, default=3, help="models in the final file")
    parser.add_argument("--out", default=os.path.join(ROOT, "models", "sequence.pt"))
    parser.add_argument("--pff", type=int, default=0, help="PFF World Cup matches to train on")
    parser.add_argument("--pff-copies", type=int, default=2, help="degraded copies per PFF match")
    parser.add_argument("--experiment", default="", choices=("", "longer-mirror", "pff", "longer", "wide"),
                        help="longer-mirror: test 2x longer training and mirrored predictions, "
                             "save held-out predictions, train no final model")
    parser.add_argument("--final-ways", default="",
                        help="comma-separated ways to train the final models with, skipping the "
                             "held-out test (e.g. 'less smoothing,sharper changes')")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dfl = sorted(glob.glob(os.path.join(ROOT, "cache", "DFL-*.npz")))
    metrica = sorted(glob.glob(os.path.join(ROOT, "cache", "MET-*.npz")))
    pff = sorted(glob.glob(os.path.join(ROOT, "cache", "PFF-*.npz")))[:args.pff]
    paths = dfl[:args.dfl or None] + metrica[:args.metrica] + pff
    if not dfl:
        raise SystemExit("no cached external matches; run scripts/prepare_external.py first")
    source = [{"DFL": "DFL", "MET": "Metrica", "PFF": "PFF"}[os.path.basename(p)[:3]]
              for p in paths]
    print("device {}, {} DFL + {} Metrica + {} PFF matches, {} copies each ({} per PFF match)".format(
        device, source.count("DFL"), source.count("Metrica"), source.count("PFF"), args.copies,
        args.pff_copies), flush=True)
    # Held-out tests are on DFL and Metrica matches only; PFF matches are only ever trained on.
    tested = [i for i, src in enumerate(source) if src != "PFF"]

    started = time.time()
    variants = list(STRESS)[:1 if args.no_stress else None]
    count = [args.pff_copies if src == "PFF" else args.copies for src in source]
    train_jobs = [(p, 100 * c + i, None) for i, p in enumerate(paths) for c in range(count[i])]
    finals_only = [w.strip() for w in args.final_ways.split(",") if w.strip()]
    test_jobs = [] if finals_only else [(paths[i], 500 + i, v) for i in tested for v in variants]
    with Pool(args.workers) as pool:
        prepared = pool.map(prepare, train_jobs + test_jobs, chunksize=1)
    train_data, test_data = prepared[:len(train_jobs)], prepared[len(train_jobs):]
    offsets = np.concatenate(([0], np.cumsum(count)))
    by_match = [train_data[offsets[i]:offsets[i + 1]] for i in range(len(paths))]
    names = train_data[0]["names"]
    print("prepared {} training copies, {} features ({:.0f} s)".format(
        len(train_data), len(names), time.time() - started), flush=True)

    # The comparison: three ways of training the same network on all the other matches.
    # "fewer merges" and "sharper changes" aim at the possessions we still merge together.
    plan = [("current", 0.15, 1.0), ("less smoothing", 0.05, 1.0), ("sharper changes", 0.15, 3.0)]

    def spell(prob, item, team=12.0, dead=32.0):
        pred = decode.decode_match(prob, item["runs"], team, dead)
        return score(item["y"].astype(np.int8), pred)

    if args.experiment in ("pff", "longer", "wide"):
        everyone = {"DFL", "Metrica", "PFF"}
        trials = {
            "pff": [("without PFF", {"DFL", "Metrica"}, None, args.cv_steps),
                    ("with PFF", everyone, None, args.cv_steps)],
            # With 74 matches there is far more to learn from: does longer training, or a
            # bigger network trained longer, now pay off? (Compare with "with PFF" from the
            # "pff" run, which used the same folds and seeds.)
            "longer": [("with PFF, 2x steps", everyone, None, 2 * args.cv_steps)],
            "wide": [("with PFF, wide, 2x steps", everyone, WIDE, 2 * args.cv_steps)],
        }[args.experiment]
        totals = {}
        for t, k in enumerate(tested):
            name = os.path.basename(paths[k])[:-4]
            print("held out {} ({})".format(name, source[k]), flush=True)
            for tag, allowed, config, steps in trials:
                copies = [c for i, group in enumerate(by_match)
                          if i != k and source[i] in allowed for c in group]
                net, mean, std = train(copies, steps, device, seed=k,
                                       label="fold {} {}".format(k, tag), config=config)
                ckpt = {"names": names, "mean": mean, "std": std}
                for j, variant in enumerate(variants):
                    item = test_data[t * len(variants) + j]
                    prob = seqmodel.predict(net, ckpt, item["match"], item["features"], device)
                    s = spell(prob, item)
                    totals.setdefault("{:<8} test, {:<26} {}".format(source[k], tag, variant),
                                      []).append(s["spell"])
                    if j == 0:
                        print("  " + line(tag, s), flush=True)
                    if j == 0 and args.experiment == "pff":
                        # The same network, also asked about the mirrored match, averaged.
                        both = seqmodel.predict_mirrored([(net, ckpt)], item["match"],
                                                         item["features"], device)
                        s = spell(both, item)
                        totals.setdefault("{:<8} test, {:<21} {}".format(
                            source[k], tag + " + mirror", variant), []).append(s["spell"])
                        print("  " + line(tag + " + mirror", s), flush=True)
                del net
        print()
        print("mean spellF1 over held-out matches (viterbi 12/32)")
        for key, values in sorted(totals.items()):
            print("  {:<52} {:.4f}  (min {:.4f}, max {:.4f}, n {})".format(
                key, float(np.mean(values)), float(np.min(values)), float(np.max(values)),
                len(values)))
        print("\n({:.0f} s total)".format(time.time() - started))
        sys.exit(0)

    if args.experiment == "longer-mirror":
        oof, totals = {}, {}
        out_dir = os.path.dirname(os.path.abspath(args.out))
        for t, k in enumerate(tested):
            name = os.path.basename(paths[k])[:-4]
            print("held out {} ({})".format(name, source[k]), flush=True)
            copies = [c for i, group in enumerate(by_match) if i != k for c in group]
            for steps in (args.cv_steps, 2 * args.cv_steps):
                net, mean, std = train(copies, steps, device, seed=k,
                                       label="fold {} {} steps".format(k, steps))
                members = [(net, {"names": names, "mean": mean, "std": std})]
                for j, variant in enumerate(variants):
                    item = test_data[t * len(variants) + j]
                    plain = seqmodel.predict_all(members, item["match"], item["features"], device)
                    both = seqmodel.predict_mirrored(members, item["match"], item["features"],
                                                     device)
                    oof["{}|y".format(name)] = item["y"].astype(np.int8)
                    oof["{}|runs".format(name)] = np.array(item["runs"], dtype=np.int64)
                    for tag, prob in (("{} steps".format(steps), plain),
                                      ("{} steps mirrored".format(steps), both)):
                        oof["{}|{}|{}".format(name, tag, variant)] = prob.astype(np.float16)
                        s = spell(prob, item)
                        key = "{:<8} test, {:<20} {}".format(source[k], tag, variant)
                        totals.setdefault(key, []).append(s["spell"])
                        if j == 0:
                            print("  " + line(tag, s), flush=True)
                del net
            np.savez_compressed(os.path.join(out_dir, "oof_longer_mirror.npz"), **oof)
        print()
        print("mean spellF1 over held-out matches (viterbi 12/32)")
        for key, values in sorted(totals.items()):
            print("  {:<60} {:.4f}  (min {:.4f}, max {:.4f}, n {})".format(
                key, float(np.mean(values)), float(np.min(values)), float(np.max(values)),
                len(values)))
        print("\nsaved {} ({:.0f} s total)".format(
            os.path.join(out_dir, "oof_longer_mirror.npz"), time.time() - started))
        sys.exit(0)

    # Held-out probabilities are saved so decoders can be compared later without a GPU.
    oof = {}
    totals, stress = {}, {}
    for t, k in enumerate([] if finals_only else tested):
        print("held out {} ({})".format(os.path.basename(paths[k]), source[k]), flush=True)
        copies = [c for i, group in enumerate(by_match) if i != k for c in group]
        for tag, smooth, boost in plan:
            net, mean, std = train(copies, args.cv_steps, device, seed=k,
                                   label="fold {} {}".format(k, tag), smooth=smooth, boost=boost)
            ckpt = {"names": names, "mean": mean, "std": std}
            for j, variant in enumerate(variants):
                item = test_data[t * len(variants) + j]
                prob = seqmodel.predict(net, ckpt, item["match"], item["features"], device)
                name = os.path.basename(paths[k])[:-4]
                oof["{}|{}|{}".format(name, tag, variant)] = prob.astype(np.float16)
                oof["{}|y".format(name)] = item["y"].astype(np.int8)
                oof["{}|runs".format(name)] = np.array(item["runs"], dtype=np.int64)
                s = spell(prob, item)
                key = "{:<8} test, {:<16} {}".format(source[k], tag, variant)
                if j == 0:
                    print("  " + line(tag, s), flush=True)
                    totals.setdefault(key, []).append(s["spell"])
                else:
                    stress.setdefault(key, []).append(s["spell"])
            del net
        np.savez_compressed(os.path.join(os.path.dirname(os.path.abspath(args.out)), "oof.npz"),
                            **oof)

    def table(title, rows):
        print()
        print(title)
        for name, values in sorted(rows.items()):
            print("  {:<52} {:.4f}  (min {:.4f}, max {:.4f}, n {})".format(
                name, float(np.mean(values)), float(np.min(values)), float(np.max(values)),
                len(values)))

    if totals:
        table("mean spellF1 over held-out matches, calibrated noise (viterbi 12/32)", totals)
    if stress:
        table("same models under other noise (viterbi 12/32)", stress)

    # Final models on every match: the requested ways, or else whichever way scored best on
    # the held-out DFL matches (they are the closest to how the competition's answers were made).
    def dfl_mean(tag):
        return float(np.mean(totals["{:<8} test, {:<16} {}".format("DFL", tag, variants[0])]))
    if finals_only:
        ways = [p for p in plan if p[0] in finals_only]
        if len(ways) != len(finals_only):
            raise SystemExit("unknown way in --final-ways; known: {}".format([p[0] for p in plan]))
    else:
        ways = [max(plan, key=lambda p: dfl_mean(p[0])) if totals else plan[0]]
    members = []
    for way, smooth, boost in ways:
        print()
        print("final models trained the '{}' way".format(way), flush=True)
        for seed in range(args.seeds):
            net, mean, std = train(train_data, args.steps, device, seed=99 + seed,
                                   label="final {} {}".format(way, seed), smooth=smooth,
                                   boost=boost)
            members.append({"state": net.cpu().state_dict(), "names": names, "mean": mean,
                            "std": std, "config": seqmodel.CONFIG, "tag": "shape",
                            "training": way})
    best = (", ".join(w[0] for w in ways),)
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    torch.save({"members": members}, args.out)
    print()
    print("saved {} with {} models, trained the '{}' way ({:.0f} s total)".format(
        args.out, len(members), best[0], time.time() - started))


## 1. Download and convert the open DFL and Metrica matches (about 15 minutes)

In [ ]:
!python scripts/download_external.py
!python scripts/prepare_external.py
!python scripts/download_metrica.py
!python scripts/prepare_metrica.py
!rm -rf external
# PFF World Cup matches, converted locally (scripts/prepare_pff.py) and attached
# to this notebook as a private Kaggle dataset; skipped if not attached.
!find /kaggle/input -name 'PFF-*.npz' -exec cp {} cache/ \;
!ls cache | grep -c PFF || true

## 2. Train

First a leave-one-match-out test (the printed `mean spellF1` is the number that matters), then the final model on all seven matches.

In [ ]:
!python scripts/train_seq.py --copies 8 --cv-steps 4000 --steps 8000 --pff 64 --pff-copies 2 --experiment longer --out /kaggle/working/sequence.pt

In [ ]:
!ls -la /kaggle/working && rm -rf /kaggle/working/repo/cache